# 04 · Análisis descriptivo, exploratorio y preprocesado

**Qué hace este notebook.** Toma el set de features leak-safe que produce
`scripts/features/build_features.py` (validado por `qa_features.py`)
y lo lleva de "features crudas correctas" a "matriz de modelado limpia y
reproducible". El recorrido es: describir → explorar la señal → entender la
estructura (colinealidad, drift, huecos) → **decidir** con criterio de analista
→ materializar esas decisiones en un **módulo reproducible** `preprocess.py`.

**Qué NO hace.** No entrena modelos. No descarta ninguna variable por rendimiento
predictivo: las podas de la Parte D son estructurales —constantes, duplicados
deterministas, colinealidad— y no consultan en ningún momento qué tan bien predice
cada columna. Quedarse con unas variables y no con otras en función de lo que
aciertan es competencia del `05_MODELADO_Y_EVALUACION.ipynb`, donde el banco de
ensayos (`scripts/models/laboratorio.py`) mide cada conjunto propuesto bajo el
protocolo de validación temporal (`scripts/models/protocol.py`) y quien lee decide.
Esa decisión se toma sobre las temporadas de desarrollo y queda registrada ensayo a
ensayo, con su sello y sus cifras; la temporada sellada no participa en ella y se lee
una sola vez, al final. Elegir el conjunto mirando el resultado sobre todas las
temporadas, holdout incluido, sería exactamente la fuga de selección que este trabajo
evita.

Una salvedad honesta: la poda colineal de D.3 sí arbitra por `MI × cobertura`, que
es un criterio supervisado. Por eso `fit` no se ejecuta una única vez para medir,
sino que el protocolo lo repite dentro de cada pliegue sobre la ventana que precede
al partido evaluado; el apartado final de este notebook desarrolla por qué.

**El contrato anti-fuga de este notebook (innegociable):**

1. **El holdout 2025/26 no decide nada.** Todo estadístico que informe una
   decisión (umbrales, constantes de imputación, encoders, escalado) se calcula
   **solo sobre `dev`** (2016/17–2024/25). El holdout se *transforma* con esos
   parámetros, jamás se *ajusta* con él.
2. **El preprocesado es una función, no una serie de celdas.** Se encapsula en
   `scripts/features/preprocess.py`, que expone `fit(dev) → PreprocessState` y
   `apply(df, state)`. El estado se aprende una sola vez en *dev*, se serializa
   (`PreprocessState.save/load`) y el **mismo** `apply` transforma `dev`, `holdout`
   y `features_upcoming` (inferencia), garantizando **paridad train↔scoring** bit a
   bit (misma filosofía que el motor compartido `build_features` / `build_scoring`).
3. **Ninguna transformación puede mirar el futuro del propio partido.** Las
   features ya son backward-looking; el preprocesado no puede reintroducir fuga
   (p. ej. imputar con la media de toda la temporada incluyendo partidos
   posteriores). Imputación siempre con constantes fijadas en `fit(dev)`.

In [ ]:
# --- Entorno y carga -------------------------------------------------------
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

# raíz del repo + acceso al módulo de preprocesado (fuente única de la verdad)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC     = ROOT / "data" / "02_processed"
FEATURES = PROC / "features_matches.parquet"
sys.path.insert(0, str(ROOT / "scripts" / "features"))

feat = pd.read_parquet(FEATURES)

# Trazabilidad publicada junto al conjunto: procedencia por columna y
# cobertura por partido. Se leen de disco -y no se recalculan- porque este
# notebook describe EL CONJUNTO QUE HAY, no el que se construiria ahora.
from lineage import coverage_by_season
lin = pd.read_parquet(PROC / "feature_lineage.parquet")
cov = pd.read_parquet(PROC / "coverage_matrix.parquet")

# separación estricta desde el minuto 1: el holdout 25/26 no participa en NINGÚN
# cálculo que informe una decisión; solo se transforma (Parte D) y se audita (E).
dev     = feat[feat["split"] == "dev"].copy()
holdout = feat[feat["split"] == "holdout"].copy()

def temporada(y):
    """año de inicio -> etiqueta 'YY/YY+1' (2016 -> '16/17')."""
    y = int(y); return f"{y % 100:02d}/{(y + 1) % 100:02d}"

print(f"features_matches: {feat.shape[0]} partidos x {feat.shape[1]} columnas")
print(f"  dev     = {len(dev):4d}  ({temporada(dev['season_year'].min())}"
      f"-{temporada(dev['season_year'].max())})")
print(f"  holdout = {len(holdout):4d}  ({temporada(holdout['season_year'].min())}, sellado)")

---
## Parte A — Análisis descriptivo (univariante)

Retrato del dataset antes de tocar nada: forma, tipos, huecos, target y
distribución de cada variable. El objetivo de esta parte es *localizar todo lo
que pueda ensuciar el dataset* y dejarlo documentado, sin decidir todavía.

### A.1 · Dimensiones, tipos e inventario de familias
Forma, memoria, dtypes. Inventario de las **familias de features** (mercado,
Elo, valor plantilla, fatiga, bajas, rotación del once, forma, h2h, clasificación,
contexto, fuerza latente, derivadas) con
nº de columnas de cada una — para razonar por bloques y no columna a columna.

La agrupación por familias es **descriptiva**: sirve para recorrer el análisis por bloques
homogéneos de escala y de forma. Convive con la agrupación por **procedencia** —qué módulo
emite cada columna y de qué fuente sale— que se publica en `feature_lineage.parquet`. Las dos
responden a preguntas distintas y pueden repartir las columnas de manera distinta, pero no
pueden discrepar sobre **qué columnas existen**: la celda comprueba la igualdad de conjuntos y
se detiene si el conjunto que se analiza no es el que la trazabilidad documenta.

In [ ]:
# --- A.1 · Dimensiones, tipos e inventario de familias ---------------------
# Cada columna se mapea a su familia por prefijo/infijo. El assert garantiza que
# NINGUNA queda sin clasificar: si el pipeline añade una feature nueva, salta aquí.
META = ["match_key","season","season_year","date","home_team","away_team",
        "home_key","away_key","match_id","round","y","y_num","split"]

def familia(col):
    if col in META: return "meta"
    c = col.lower()
    # El mercado se comprueba primero porque su bloque incluye las cuotas crudas de
    # SERVICIO -las que no son variables pero viajan en la matriz para que el
    # backtest liquide con ellas- y conviene que queden contadas junto al resto del
    # precio y no repartidas por coincidencia de token.
    if any(t in c for t in ("odds", "mkt_", "ah_", "overround", "close_move",
                            "sharp_div", "b365")): return "mercado"
    # Una columna se clasifica por lo que MIDE, no por la materia prima que emplea.
    # La dureza del calendario reciente (`form_opp_elo_z_*`) se construye con la
    # fuerza tipificada de ClubElo, pero lo que describe es contra quien ha jugado el
    # equipo, no su propio nivel: pertenece a "forma". La comprobacion va ANTES que la de "elo" porque su
    # nombre contiene el token y, de lo contrario, quedaria clasificada por la
    # herramienta y no por el concepto.
    # La fuerza latente se comprueba antes que nada porque su nombre no comparte
    # token con ninguna otra familia y porque lo que mide -las dos componentes que el
    # Elo funde en un escalar- es un concepto propio: clasificarla bajo "elo" por la
    # vecindad conceptual escondería que son estimaciones distintas, de fuente
    # distinta (el marcador) y con un modo de ausencia distinto.
    if "strength" in c: return "fuerza"
    if "opp_elo" in c: return "forma"
    if "elo" in c: return "elo"
    # Las bajas se comprueban ANTES que el valor de plantilla: el peso economico de
    # una ausencia (`home_out_value_pct`) contiene el token del valor y, con el orden
    # inverso, quedaria clasificado por la magnitud que emplea para ponderar en lugar
    # de por lo que mide, que es quien falta.
    if any(t in c for t in ("_out", "out_value", "out_minshare",
                            "n_missing")): return "bajas"
    if any(t in c for t in ("squad_value", "age_diff")): return "valor"
    if any(t in c for t in ("fat_", "matches_prev")): return "fatiga"
    # La rotacion es familia propia y no una extension de la fatiga, aunque las dos
    # describan como llega el equipo. La fatiga cuenta PARTIDOS -descanso y congestion
    # del calendario- y la rotacion cuenta MINUTOS y continuidad del once: dos
    # denominadores distintos sobre el mismo suceso, y fundirlos esconderia justo el
    # matiz que el bloque anade, que un suplente y un titular no acumulan el mismo
    # desgaste en el mismo numero de encuentros.
    if any(t in c for t in ("xi_continuity", "starters_minutes",
                            "minutes_hhi")): return "rotacion"
    if "h2h" in c: return "h2h"
    if any(t in c for t in ("playstyle", "effectiveness", "relative_perf",
                            "post_big")): return "psico"
    # La trascendencia se declara junto a la clasificacion porque de ella sale: son
    # lecturas de la misma tabla -donde esta un equipo, que le queda por jugar y que
    # puede todavia ganar o perder-. Sus nombres se enumeran con la RAIZ COMPLETA y no
    # por el token suelto `points`, y la precision no es cosmetica: la forma reciente
    # publica `form_points_*`, `cum_points_pg_diff` y `form_xpts_*`, de modo que un
    # token generico se llevaria diez columnas de forma a esta familia y dejaria el
    # inventario contando once bloques que no describen la tabla.
    if any(t in c for t in ("prev_position", "position_zone", "dynamic_pos",
                            "points_above_drop", "points_to_", "safe_from_drop",
                            "matches_left", "stakes_none")): return "clasificacion"
    # La calidad de ocasion se declara explicitamente aunque el atajo generico de la
    # linea siguiente la acertaria por contener el token `form`: lo que estas columnas
    # miden es cuanto vale el disparo corriente, propio y concedido, y ese es el
    # motivo de que pertenezcan a "forma", no la coincidencia del nombre.
    if "form_shot_" in c: return "forma"
    if any(t in c for t in ("form", "cum_", "xg", "npxg", "ppda", "deep",
                            "xpts")): return "forma"
    # El contexto se comprueba al final porque describe DONDE y CUANDO se juega
    # -calendario y geografia- y no como llegan los equipos: sus nombres no
    # comparten token con ninguna otra familia, de modo que ninguna comprobacion
    # anterior puede reclamarlo y el orden aqui no cambia el reparto.
    if any(t in c for t in ("kickoff", "weekend", "travel_km",
                            "derby")): return "contexto"
    return "otras"

fam = pd.Series({c: familia(c) for c in feat.columns})
otras = fam[fam == "otras"].index.tolist()
assert not otras, f"columnas sin familia (revisar mapeo): {otras}"

print(f"{feat.shape[0]} partidos x {feat.shape[1]} columnas  "
      f"({feat.memory_usage(deep=True).sum()/1e6:.1f} MB)")
print("\ntipos:", dict(feat.dtypes.astype(str).value_counts()))
print("\ncolumnas por familia:")
print(fam.value_counts().to_string())
# La familia de arriba es una agrupacion DESCRIPTIVA, pensada para recorrer el
# analisis por bloques homogeneos de forma y de escala. La procedencia publicada
# agrupa con otro criterio -por modulo que emite la columna- y las dos
# convivencias son legitimas mientras no se contradigan. Lo que no puede pasar es
# que una columna exista aqui y no alli: eso significaria que el conjunto que se
# esta analizando no es el que la memoria documenta.
_declaradas = set(lin.columna)
_reales = set(feat.columns) - set(META)
print("\ncolumnas del conjunto sin procedencia declarada:",
      sorted(_reales - _declaradas)[:8] or "ninguna")
print("columnas declaradas ausentes del conjunto      :",
      sorted(_declaradas - _reales)[:8] or "ninguna")
assert not (_reales - _declaradas) and not (_declaradas - _reales), \
    "el conjunto y la tabla de procedencia han divergido: reconstruya features"

# Correspondencia entre las dos agrupaciones. Se muestra para que quede escrito
# que la familia descriptiva no inventa bloques: cada una se reparte entre unos
# pocos bloques de procedencia, y las diferencias son las esperadas (la forma
# ajustada al rival usa ClubElo pero mide forma; el contexto sale del sello de
# saque pero describe el encuentro).
_cruce = (pd.crosstab(fam.reindex(lin.columna).values, lin.bloque.values)
            .rename_axis(index="familia descriptiva", columns="bloque declarado"))
_cruce.loc[:, _cruce.sum() > 0]

### A.2 · Mapa de valores faltantes (missing) por grupo
La mayor parte de las columnas-feature tiene algún NaN en *dev* —el recuento exacto
lo imprime la celda, y crece con el conjunto—, pero **casi ninguno es aleatorio**: cada hueco responde a una causa estructural, y esa causa —no el %
en bruto— decide la imputación en la Parte D. El recuento por causa (arriba) dice
*cuántas y por qué*; los tres heatmaps lo enseñan: variables en el eje vertical
(para que se lean) y temporadas en el horizontal, con el **rojo = NaN** formando
**bloques** que delatan el origen estructural.

- **G1 · Mercado.** Bloque rojo nítido arriba: las cuotas de cierre,
  hándicap asiático, over/under y la deriva de la línea (`close_move_*`) están **100% ausentes en 16/17–18/19
  y presentes al 100% desde 19/20** (football-data no publicaba esos mercados). Es
  el hueco dominante = 3 temporadas de *dev*. El resto (prob. implícitas `mkt_p_*`,
  cuotas de apertura, Pinnacle `odds_ps_*`) está completo.
- **G2 · Forma / clasificación / estilo.** Domina una **fina franja roja
  al inicio de cada temporada**: la jornada 1 sin forma ni clasificación previa
  (~2.7% en casi todas). Sobresalen dos filas: `form_big_chances`
  (100% → 39% → 3% en las dos primeras temporadas) y
  `venue_form` (~5% fijo, poca historia local/visitante al arrancar cada curso).
  Las lecturas de la memoria comparten esa franja y solo esa: la ventana corta, el
  decaimiento exponencial, el acumulado de temporada y la dureza del calendario se
  calculan sobre el mismo historial desplazado, así que faltan exactamente cuando no hay
  partido previo. La dureza del calendario encadena además ClubElo, pero ClubElo cubre el
  censo completo de clubes y no añade hueco propio; si alguna vez lo añadiera,
  `build_scoring` lo vigila en un bloque de cobertura separado precisamente para que no
  pase inadvertido. La
  **fuerza latente** se dibuja en este mismo grupo, y no con los priores de G3, porque
  su patrón de faltante es el de la forma y no el de un dato siempre disponible; pero es
  un patrón propio y algo más ancho que la franja de la jornada 1, porque el bloque exige
  tres partidos efectivos de historia y describe un **enfrentamiento**: basta con que a
  uno de los dos clubes le falte recorrido para que el partido entero se quede sin él.
  De ahí que su hueco se cuente como causa aparte y estrene flag propio en D.2.
  La **rotación del once** (`xi_continuity`, `starters_minutes_*`, `minutes_hhi`) se
  dibuja también aquí, y por el mismo criterio: se calcula sobre los partidos anteriores
  del equipo, de modo que falta exactamente donde falta la forma —el primer encuentro de
  cada equipo y temporada— y su rojo dialoga con esa franja y no con el gris de los
  priores. Que comparta patrón con la forma es además la comprobación de que el bloque
  mira solo hacia atrás: si describiera el partido a predecir, estaría siempre completo.
  La **calidad de ocasión** (`form_shot_xg_mean_*` y su gemela concedida,
  `form_shot_xg_mean_against_*`) se dibuja también en este grupo y admite dos lecturas muy distintas del
  mismo rojo. Si el volcado de tiros se ha ejecutado, su patrón es el de la forma más un
  residuo propio: el agregado excluye penaltis y goles en propia, de modo que un lado sin
  remates admisibles no aporta nada a la ventana aunque el partido esté cubierto, y por
  eso su hueco se cuenta como causa aparte. Si no se ha ejecutado, sus filas aparecen
  **enteras en rojo**: es un bloque opcional ausente, no un faltante estructural, y no
  llega a la imputación porque la poda por constancia de `fit(dev)` retira antes
  cualquier columna sin variación. Distinguir a simple vista las dos lecturas es la razón
  de dibujarlas junto a la franja y no en un grupo propio.
- **G3 · Fuerza y contexto.** Elo, valor de plantilla y fatiga están
  **completos** (gris). El rojo se concentra en dos sitios: **bajas** — bloque en
  16/17 (no se recogían) + un residual del ~11–17% en los recuentos `n_*`. Dentro de ese
  bloque, la **cuota de minutos** de los ausentes (`out_minshare_*`) añade un hueco propio
  que no comparte con el recuento ni con el valor: pesar una baja por el papel que venía
  teniendo exige que sus compañeros hayan acumulado minutos, y en las primeras jornadas de
  cada temporada no existe todavía ese denominador aunque el parte médico sí exista. Se
  cuenta por eso como causa distinta, para no atribuir al extractor de bajas un hueco que
  es de la aritmética de la ponderación; y **H2H**
  (`h2h_gd_ewm`) — *ausencia real*: 50% en 16/17 que **decae hacia 0** a medida que
  se llena el histórico de enfrentamientos. El **contexto** —franja de saque,
  fin de semana, distancia de viaje y derbi— se dibuja en este mismo grupo y aparece
  **gris**: el sello de saque viaja en el máster para todo el censo de partidos y la
  tabla de estadios cubre el censo completo de clubes, de modo que el único hueco que
  admite es de *cobertura* —un club sin ficha de estadio o un partido sin hora
  publicada— y no de época ni de arranque de temporada. `build_features` lo vigila con
  `unknown_stadiums`, que enumera los clubes sin coordenadas para que ese hueco, si
  aparece, no pase inadvertido.

El **valor de plantilla** sale completo aquí porque el parquet aún es *pre-lag*; al
regenerarlo con el desfase t-1 (D.0) los ascendidos pasan a NaN y los absorbe
`weak_prior` (D.2). Cada bloque → su estrategia de imputación y su flag propio.

Conviene subrayar qué NO aparece en estos mapas: las columnas que el catálogo no
declara. El recuento de faltantes es una de las cosas que la curación de la capa de
features vuelve legible, porque un mapa que enumera cuatrocientas columnas —muchas de
ellas reparametrizaciones de la misma— dibuja el mismo hueco varias veces y hace parecer
estructural lo que solo era repetición.

In [ ]:
# --- A.2 · Mapa de valores faltantes (missing) por GRUPO -------------------
# La mayoría de features tiene algún NaN en dev, pero casi ninguno es aleatorio.
# Se cuenta por CAUSA (decide la imputación en D.2) y se dibuja en 3 heatmaps por
# grupo temático: variables en el eje Y (legibles) y temporada en el X, de modo
# que el faltante se ve como BLOQUES (estructural), no disperso. SOLO dev.
import seaborn as sns

na = (dev.isna().mean() * 100); na = na[na > 0]
# El reparto por causa refleja UNO A UNO las estrategias de imputacion de D.2:
# "epoca" y el cold-start generico se estiman (neutral), la ausencia real se
# resuelve a cero (zero), el ascendido toma un prior debil (weak_prior) y la
# trascendencia se rellena con su valor cierto (known). Leer el mapa aqui y decidir
# alli con otras categorias abriria una grieta entre lo que se describe y lo que se
# hace, que es justo lo que este cuaderno esta para evitar.
def causa(col):
    c = col.lower()
    if any(t in c for t in ("close", "ah_", "over25", "under25", "odds_ps")):     return "época · mercado 16-19"
    if any(t in c for t in ("overround", "mkt_spread", "sharp_div")):             return "época · desglose por casa"
    # El orden importa: la cuota de minutos lleva el token de las bajas y, sin esta
    # comprobacion delante, quedaria contada dentro del hueco de epoca. Son causas
    # distintas: la de epoca es una temporada entera sin parte medico, esta es la
    # falta de denominador -nadie ha acumulado minutos- al arrancar el curso.
    if "out_minshare" in c:                                                        return "cold-start · sin minutos acumulados"
    if any(t in c for t in ("out", "n_missing")):                                  return "época · bajas 16/17"
    # La rotacion falta donde falta la forma -no hay partido anterior del que leer el
    # once-, pero se nombra aparte para que el recuento no la esconda dentro de un
    # bloque con el que no comparte ni fuente ni modulo.
    if any(t in c for t in ("xi_continuity", "starters_minutes", "minutes_hhi")):  return "cold-start · sin partido anterior (rotación)"
    # La calidad de ocasion comparte franja con la forma pero no comparte causa. A la
    # jornada 1 se le suma un motivo que solo tiene este bloque: el agregado excluye
    # penaltis y goles en propia, de modo que un lado puede quedarse sin ningun remate
    # admisible en un partido por lo demas cubierto y no aportar nada a la ventana.
    if "form_shot_" in c:                                                          return "cold-start · sin remates previos admisibles"
    if "h2h" in c:                                                                 return "real · sin H2H previo"
    # El valor de plantilla y la edad son las dos lecturas que quedan del bloque de
    # Transfermarkt tras la curacion del catalogo, y las dos faltan por el mismo
    # motivo: un ascendido no tiene temporada t-1 en Primera de la que leerlas.
    if any(t in c for t in ("squad_value", "age_diff")):                           return "cold-start · ascendido t-1"
    if "venue_form" in c:                                                          return "real · sin partidos en ese campo"
    if any(t in c for t in ("kickoff", "weekend", "travel_km", "derby")):          return "cobertura · estadio u hora sin dato"
    # La fuerza latente falta por una causa que no es la de la jornada 1: el bloque
    # pide tres partidos EFECTIVOS de historia en la categoria y se lo niega al
    # partido entero si a uno de los dos clubes le faltan. Contarla con el cold-start
    # generico la haria desaparecer dentro de la franja de inicio de temporada, que es
    # justamente la lectura equivocada: su hueco persiste jornadas despues del arranque
    # para un ascendido y no aparece en absoluto para el resto.
    if "strength" in c:                                                            return "cold-start · sin recorrido en la categoría"
    if any(t in c for t in ("points_to_", "points_above_drop", "safe_from_drop",
                            "stakes_none", "matches_left")):                       return "cold-start · trascendencia (valor cierto)"
    return "cold-start · forma/tabla jornada 1"
print(f"columnas con algún NaN en dev: {len(na)}  (de {feat.shape[1]-len(META)} features)")
print("\nNaN por CAUSA (nº de columnas):")
print(pd.Series({c: causa(c) for c in na.index}).value_counts().to_string())

# orden cronológico + límites de temporada para el eje X de los heatmaps
dsort = dev.sort_values(["date","match_key"]).reset_index(drop=True)
sy = dsort["season_year"].values
bounds  = [0] + [i for i in range(1,len(sy)) if sy[i] != sy[i-1]] + [len(sy)]
centers = [(bounds[k]+bounds[k+1])//2 for k in range(len(bounds)-1)]
etiq    = [temporada(sy[bounds[k]]) for k in range(len(bounds)-1)]

def grupo(c):
    x = c.lower()
    if any(t in x for t in ("odds", "mkt_", "ah_", "overround", "close",
                            "sharp_div", "b365")):
        return 0
    # Mismo criterio que familia(): la dureza del calendario reciente se dibuja con
    # la forma (G2) y no con el Elo (G3). Su patrón de faltantes es el de la forma
    # -franja de jornada 1-, no el de un prior de fuerza siempre disponible;
    # mezclarla en G3 rompería la lectura por bloques que da sentido al heatmap.
    if "opp_elo" in x:
        return 1
    # La fuerza latente va tambien a G2, y por el mismo criterio: el heatmap agrupa por
    # PATRON de faltante y no por parentesco conceptual. Aunque describa un prior de
    # fuerza -como el Elo de G3-, se estima con el historial de resultados y falta
    # cuando ese historial no existe, de modo que su rojo dialoga con la franja de
    # arranque de temporada y no con el gris de los datos siempre disponibles.
    if "strength" in x:
        return 1
    # La rotacion se queda en G2 por decision explicita y no por caer en el defecto:
    # su patron de faltante es la franja de arranque de temporada, igual que el de la
    # forma, mientras que G3 reune lo que esta disponible con semanas de antelacion.
    if any(t in x for t in ("xi_continuity", "starters_minutes", "minutes_hhi")):
        return 1
    # La calidad de ocasion se dibuja en G2 por decision expresa: se calcula sobre el
    # historial desplazado del equipo y su rojo es por tanto la franja de arranque de
    # temporada, la misma que la de la forma, y no el gris de los priores de G3.
    if "form_shot_" in x:
        return 1
    if any(t in x for t in ("elo", "squad_value", "age_diff", "fat_",
                            "matches_prev", "out", "n_missing", "h2h")):
        return 2
    # El contexto acompana a G3 y no a G2: como el Elo o el valor de plantilla, es
    # un dato disponible con semanas de antelacion y ajeno al estado de forma, de
    # modo que su patron de faltantes -practicamente ninguno- se lee junto al de
    # los priores siempre disponibles y no junto a la franja de la jornada 1.
    if any(t in x for t in ("kickoff", "weekend", "travel_km", "derby")):
        return 2
    return 1
GRUPOS = {0: [], 1: [], 2: []}
for c in dsort.columns:
    if c not in META: GRUPOS[grupo(c)].append(c)
titulos = ["G1 · Mercado (cuotas y derivadas)",
           "G2 · Forma / clasificación / estilo",
           "G3 · Fuerza y contexto (Elo · valor · fatiga · bajas · H2H · calendario y geografía)"]

for k in range(3):
    cols = sorted(GRUPOS[k], key=lambda c: dsort[c].isna().mean(), reverse=True)
    fig, ax = plt.subplots(figsize=(15, max(3, len(cols)*0.13)))
    sns.heatmap(dsort[cols].isna().T, cbar=False, cmap=["#e9eef2","#c62828"],
                yticklabels=cols, xticklabels=False, ax=ax)
    ax.set_yticklabels(cols, fontsize=6, rotation=0)
    for b in bounds[1:-1]: ax.axvline(b, color="white", lw=0.8)
    ax.set_xticks(centers); ax.set_xticklabels(etiq, fontsize=8)
    ax.set_title(f"{titulos[k]}  —  rojo = NaN   ({len(cols)} variables)", fontsize=10)
    plt.tight_layout(); plt.show()

### A.3 · Duplicados e integridad de claves
`match_key` único (ya garantizado por QA, se re-verifica aquí). Sin filas
duplicadas. Rango de fechas y nº de partidos por temporada = 380 (20×38) salvo
temporadas incompletas.

In [ ]:
# --- A.3 · Duplicados e integridad de claves -------------------------------
print("match_key único:", feat["match_key"].is_unique)
print("filas duplicadas completas:", int(feat.duplicated().sum()))
pm = feat.assign(t=feat["season_year"].map(temporada)).groupby("t").size()
print("\npartidos por temporada (esperado 380 = 20 equipos x 38 jornadas):")
print(pm.to_string())
print(f"\nrango de fechas: {feat['date'].min()}  ->  {feat['date'].max()}")
print("temporadas:", feat["season_year"].nunique(),
      f"({temporada(feat['season_year'].min())}-{temporada(feat['season_year'].max())})")

### A.4 · Distribución del objetivo (1X2)
Reparto global en dev: **H 45.1 % · D 26.4 % · A 28.5 %** — la ventaja de local es
el prior más fuerte del fútbol y fija el baseline a batir. Por temporada la cuota de
local oscila entre **41.6 % (20/21, COVID sin público)** y **47.9 % (22/23)**: hay
ruido año a año, pero no un *drift* sostenido que invalide el prior. El empate se
mantiene estable en torno al 26 %.

In [ ]:
# --- A.4 · Distribución del objetivo 1X2 -----------------------------------
# y: 1=gana local, X=empate, 2=gana visitante. La ventaja de campo es el prior
# más fuerte del fútbol; cualquier modelo debe al menos respetar este reparto.
order = ["H","D","A"]  # H=gana local(1), D=empate(X), A=gana visitante(2)
glob = dev["y"].value_counts(normalize=True).reindex(order)
print("reparto 1X2 en dev (%):")
print((glob*100).round(1).to_string())

by = (dev.assign(t=dev["season_year"].map(temporada))
         .groupby("t")["y"].value_counts(normalize=True)
         .unstack().reindex(columns=order))
fig, ax = plt.subplots(figsize=(9,4))
(by*100).plot(kind="bar", stacked=True, ax=ax,
              color={"H":"#2e7d32","D":"#9e9e9e","A":"#c62828"})
ax.set_ylabel("% de partidos"); ax.set_xlabel("")
ax.set_title("Distribución 1X2 por temporada (dev)")
ax.legend(title="resultado", bbox_to_anchor=(1.01,1))
plt.tight_layout(); plt.show()

### A.5 · Univariante numérico por familia
Cada familia numérica se retrata con su propia rejilla de histogramas sobre las
variables más representativas de las formas que conviven en ella. El criterio de
selección lo marca el propio gráfico descriptivo: se omiten los gemelos home/away
(comparten forma espejo) y se eligen columnas que cubran los tres patrones presentes en
los datos — **niveles** asimétricos, **contrastes `*_diff`** centrados en 0 y **colas**
de eventos raros. Cada panel anota su asimetría (skew) y su %NaN.

La tabla siguiente resume las variables seleccionadas (media, σ, rango, skew y % de
cola `|z|>4`) junto al chequeo de dominio. **Regla de oro sobre outliers**: los valores
extremos son registros históricos reales y no se recortan; solo se marca lo *físicamente
imposible*, y el chequeo de dominio (tasas ∈ [0,1], sus diferencias ∈ [−1,1], edad ∈
[15,40]) devuelve **ninguno**.

In [ ]:
# --- A.5 . Univariante numerico por familia: resumen + utilidad de dibujo ----
# Seleccion por familia: se omiten gemelos home/away y se cubren los tres patrones
# (niveles asimetricos, contrastes *_diff simetricos, colas de eventos raros).
import math
num = dev.select_dtypes("number").drop(
    columns=[c for c in ("y_num","match_id","season_year","round") if c in dev], errors="ignore")

SEL = {
 "mercado":       ["mkt_p_home","mkt_p_draw","mkt_p_away","odds_avg_h","odds_avg_a",
                   "odds_over25","odds_under25","ah_line",
                   "mkt_overround_open","mkt_spread_mean","sharp_div_home",
                   "close_move_fav"],
 # el par de fuerzas TIPIFICADAS y su momento. El nivel crudo (`home_elo`) se queda
 # en el master como columna descriptiva y no entra en la matriz: los dos tramos del
 # empalme de ClubElo no son comparables en nivel (notebook 02, seccion 6). La resta
 # se deduce del par (regla 2).
 "elo":           ["home_elo_z","away_elo_z","elo_mom_5_z_diff"],
 "valor":         ["home_squad_value","away_squad_value","age_diff"],
 # se incluye deliberadamente una variable de CADA lectura de la memoria: ventana
 # corta (_3), decaimiento exponencial (_ewm), acumulado de temporada (cum_),
 # rendimiento restringido al escenario (venue_form_) y ajuste por dureza del
 # calendario (opp_elo_z), para que la rejilla ensene que no comparten forma de la
 # distribucion. Del nucleo se retrata ademas el NIVEL (home_form_*) y no solo el
 # contraste, porque son los seis atributos de los que el catalogo emite el par.
 "forma":         ["home_form_points_ewm","home_form_gf_ewm","home_form_npxg_ewm",
                   "form_points_3_diff","form_points_ewm_diff","form_ppda_ewm_diff",
                   "form_deep_ewm_diff","form_rating_ewm_diff","form_opp_elo_z_ewm_diff",
                   "cum_xgd_pg_diff","venue_form_gd_5_diff",
                   # la CALIDAD de la ocasion, propia y concedida: una magnitud
                   # acotada y pequena (xG por remate) que no comparte forma de la
                   # distribucion ni con el volumen ni con el xG acumulado. Si el
                   # volcado de tiros no se ha ejecutado, la seleccion las omite
                   # sola: `sel_all` filtra por presencia en el conjunto.
                   "form_shot_xg_mean_ewm_diff","form_shot_xg_mean_against_ewm_diff"],
 # las tres ultimas pertenecen al bloque de TRASCENDENCIA, que se retrata aqui por
 # salir de la misma tabla: una cuenta jornadas, otra mide una distancia y la tercera
 # es un indicador, y sus formas -escalon, cola y binaria- son las que justifican
 # tratarlas distinto en D.2.
 "clasificacion": ["cum_points_pg_diff","prev_position_diff","points_above_drop_diff",
                   "home_matches_left","safe_from_drop_diff"],
 # el h2h se retrata entero: son las tres columnas que quedan del bloque. Dos son
 # contrastes del par de equipos y la tercera, la tasa de empate, es magnitud del
 # ENFRENTAMIENTO y por eso no lleva rotulo de lado.
 "h2h":           ["h2h_win_rate_ewm_diff","h2h_gd_ewm_diff","h2h_draw_rate_ewm"],
 "fatiga":        ["home_fat_rest_days","home_fat_games_last_14","matches_prev_diff"],
 "bajas":         ["home_n_missing","home_out_minshare","home_out_minshare_max",
                   "home_out_value_pct"],
 # la rotacion se retrata entera y aparte de la fatiga: cuenta MINUTOS, no partidos,
 # y sus tres columnas viven en escalas distintas -una proporcion de once repetido,
 # un acumulado de minutos de titulares y un indice de concentracion-.
 "rotacion":      ["home_xi_continuity","home_starters_minutes_14","home_minutes_hhi"],
 "psico":         ["home_post_big_win","effectiveness_diff","relative_perf_diff"],
 # el contexto se retrata ENTERO y no por muestra: son tres columnas numericas y
 # ninguna tiene gemelo local/visitante, porque describen el encuentro -donde y
 # cuando se juega- y no a un equipo. La franja `kickoff_slot` es categorica y se
 # retrata en A.6 junto a las otras dos ordinales; la hora exacta de saque ya no se
 # construye, porque la franja es esa misma informacion agrupada (regla 1).
 "contexto":      ["is_weekend","away_travel_km","is_derby"],
 # la fuerza se retrata en sus dos escalas: los dos parametros latentes
 # -logaritmicos y centrados en 0 por la normalizacion- y la terna de probabilidades
 # que el modelo de goles publica, acotada en [0,1] y que por eso queda fuera de la
 # simetrizacion de D.4.
 "fuerza":        ["strength_attack_diff","strength_defence_diff",
                   "strength_p_home","strength_p_draw","strength_p_away"],
}

sel_all = [c for fam_k in SEL for c in SEL[fam_k] if c in num.columns]
tab = num[sel_all].describe().T[["mean","std","min","50%","max"]]
tab["skew"]   = num[sel_all].skew()
z = (num[sel_all] - num[sel_all].mean()) / num[sel_all].std()
tab["%|z|>4"] = (z.abs() > 4).mean() * 100
tab.insert(0, "familia", [next(f for f in SEL if c in SEL[f]) for c in sel_all])
print(f"resumen de {len(sel_all)} variables seleccionadas por familia (dev):")
print(tab.round(3).to_string())

# imposible = fuera de DOMINIO, no "extremo real". Una tasa vive en [0,1], pero su
# *_diff (local-visitante) vive en [-1,1]; aplicar [0,1] a las diferencias marcaria
# como imposible cualquier resta negativa (falso positivo).
bad = {}
for c in [c for c in num.columns if c.startswith("mkt_p") or "strength_p_" in c
          or "rate" in c]:
    lo, hi = (-1-1e-6, 1+1e-6) if "diff" in c else (-1e-6, 1+1e-6)
    o = int(((num[c] < lo) | (num[c] > hi)).sum())
    if o: bad[c] = o
# De la edad solo llega la DIFERENCIA entre las dos plantillas, asi que el dominio
# que se comprueba no es el de una edad sino el de una resta de edades: una brecha
# de mas de diez anos entre dos plantillas de Primera no existe, y una de quince
# seria con seguridad un fallo de emparejamiento y no un club insolitamente joven.
for c in [c for c in num.columns if c == "age_diff"]:
    o = int(((num[c] < -15) | (num[c] > 15)).sum())
    if o: bad[c] = o
print("\nvalores fuera de dominio (probabilidades y tasas in [0,1], sus *_diff in "
      "[-1,1], age_diff in [-15,15]):",
      bad if bad else "ninguno")

def plot_familia(nombre):
    """Rejilla de histogramas de las variables seleccionadas de una familia."""
    cols = [c for c in SEL[nombre] if c in num.columns]
    ncol = 3; nrow = math.ceil(len(cols)/ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 2.6*nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, cols):
        s = num[c].dropna()
        ax.hist(s, bins=40, color="#1565c0", alpha=.85)
        ax.set_title(f"{c}\nskew={s.skew():.2f}  %NaN={num[c].isna().mean()*100:.1f}", fontsize=8)
        ax.tick_params(labelsize=7)
    for ax in axes[len(cols):]: ax.axis("off")
    fig.suptitle(f"A.5 - Univariante numerico . familia {nombre.upper()}", fontsize=11)
    plt.tight_layout(); plt.show()


**Mercado.** Las probabilidades implícitas están acotadas y son moderadas (`mkt_p_home` skew 0.24), con la del empate sesgada a la izquierda (`mkt_p_draw` −1.24: nunca se paga muy alto). Las cuotas en crudo muestran **cola derecha real** (`odds_avg_h` 3.45, `odds_avg_a` 3.40: favoritos extremos), mientras el over/under y el hándicap asiático son casi simétricos (`ah_line` −0.21). El movimiento de línea se concentra en torno a 0.
La **estructura del precio** se comporta como impone su construcción: el margen
(`mkt_overround_open`) vive en una banda estrecha por encima de 1 —la guarda anula la terna fuera de
[0.95, 1.60]—, mientras que el desacuerdo entre casas (`mkt_spread_mean`) y la divergencia con Pinnacle
(`sharp_div_home/draw/away`) son **no negativas y con cola derecha**: en la mayoría de partidos el mercado está de
acuerdo consigo mismo y solo unos pocos rompen el consenso. Esa forma —masa junto a cero y cola
larga— es justo el perfil que la simetrización de D.4 está pensada para tratar.

In [ ]:
# A.5 . familia MERCADO
plot_familia("mercado")


**Elo.** Del bloque viaja el **par tipificado dentro de la jornada**
(`home_elo_z`, `away_elo_z`: cuantas desviaciones separan al club de la media de la
liga ese dia) y no el nivel en puntos de rating. La razon no es de escala sino de
comparabilidad: la serie publicada empalma dos vias de ClubElo que miden lo mismo en
escalas distintas, y en nivel el empalme seria un escalon de fuerza simultaneo para
los veinte clubes (notebook 02, seccion 6). El nivel crudo se conserva en el master
como columna descriptiva y auditable, de donde B.3 lo recupera para retratar la
calibracion del prior.

La tipificacion no deshace la **forma** de la distribucion, solo su escala: el par
sigue sesgado a la derecha (skew = 0.96, pocos clubes de elite estirando la cola
hasta +2.9 desviaciones mientras el suelo se queda en -1.9), lo cual es coherente con
que la media de la liga este arrastrada hacia arriba por tres o cuatro equipos. El
momentum a 5 partidos (`elo_mom_5_z_diff`) es simetrico y pequeno, y medirlo sobre la
`z` es lo que impide que el rebase de escala del 18-10-2022 aparezca como un salto de
forma de todos los clubes a la vez.

Del par no se emite la resta: se deduce sin perder nada, y la puntuacion esperada que
de ella se derivaba (`elo_exp_home`) era una transformacion monotona de esa misma
resta --correlacion 1-- de modo que entregarla equivalia a decir tres veces lo mismo.
La logistica sigue existiendo en `prematch_derived`, y B.3 la emplea para retratar la
calibracion del prior; lo que ya no existe es la columna.


In [ ]:
# A.5 . familia ELO
plot_familia("elo")


**Valor.** El valor de plantilla tiene **cola derecha fuerte** (`home_squad_value`
1.84: Madrid y Barça tiran del extremo); es exactamente el perfil que la simetrización
de D.4 está pensada para tratar, y la razón de que baste con el par —la log-razón que
antes lo simetrizaba a mano era una función de esas dos columnas, no información nueva—.
De la edad viaja la **diferencia** (`age_diff`), casi simétrica, porque lo que
discrimina no es tener una plantilla joven sino ser más joven que el rival. El % de
valor ausente se sesga a la derecha (`home_out_value_pct` 1.14: normalmente pocas bajas
de peso).

In [ ]:
# A.5 . familia VALOR
plot_familia("valor")


**Forma** (sigue siendo la familia más grande del conjunto, aunque ya no por tres: de
cada atributo se conservan **dos** parametrizaciones —una ventana corta y el decaimiento
exponencial— y del nivel absoluto solo el del núcleo de seis atributos de resultado y
peligro, que es donde la magnitud discrimina además del contraste). Los volúmenes
ofensivos y los indicadores de estilo se sesgan a la derecha, mientras que los
**balances** quedan **centrados** y sus `*_diff` son simétricos. El rating medio es
compacto en torno a 6.8. Los números exactos los imprime la tabla de la celda anterior.

La rejilla incluye a propósito una variable de cada lectura. `form_points_ewm_diff`
reproduce el mismo balance de puntos que `form_points_3_diff`, pero ponderando de forma
continua: comparte el centrado y estrecha la dispersión, porque su memoria efectiva se
extiende más allá de la ventana aunque su semivida sea de tres partidos.
`cum_xgd_pg_diff` resume la temporada entera en vez de las últimas jornadas, y por eso
su dispersión se estrecha según avanza el curso. `venue_form_gd_5_diff` restringe el
historial al escenario —el local en su campo, el visitante fuera del suyo—, de modo que
consume el doble de partidos para llenar la misma ventana y hereda una franja de
faltante más ancha. Y `form_opp_elo_z_ewm_diff` no describe al equipo sino a sus rivales
recientes: es la fuerza media del calendario recorrido, en desviaciones. A diferencia del
par de niveles, del que hereda la materia prima, esta columna sale **simétrica**
(skew = -0.07): es un contraste entre dos calendarios, y la cola derecha que sesga la
distribución de fuerza se cancela al restar.

La lectura que **ya no está** es el residuo frente a lo que el Elo previo hacía esperar
(`form_elo_surprise_*`). No se ha quitado por redundante sino por lo que medía: la
desviación entre el resultado obtenido y el esperado es, por definición, la parte del
rendimiento que no se reproduce, y una variable cuya definición es "lo que no se explica"
no puede pedirle al modelo que la proyecte hacia adelante. Lo que sí sobrevive del ajuste
al rival es la dureza del calendario, que no mide desviación sino contra quién se jugó.

In [ ]:
# A.5 . familia FORMA
plot_familia("forma")


**Clasificación.** De este bloque solo viaja la **diferencia** entre los dos equipos: el catálogo no construye la lectura por lado porque lo que discrimina no es ir undécimo sino ir por delante o por detrás del rival. Las tres formas que dibuja la rejilla son por tanto las de una resta —`prev_position_diff`, `points_above_drop_diff` y `cum_points_pg_diff`—, simétricas en torno al cero por construcción, de modo que en ellas no se lee el sesgo sino el **grosor de las colas**: cuántos partidos enfrentan a equipos de zonas realmente distintas. La única lectura por lado del bloque es la zona de clasificación (`home_position_zone` / `away_position_zone`), que es ordinal y se analiza con las categóricas.

La rejilla incorpora además dos lecturas del bloque de **trascendencia**, que sale de
esa misma tabla. `home_matches_left` es la única columna del bloque que se emite por
lado, y con una basta: los dos equipos de un partido tienen exactamente las mismas
jornadas por delante, así que su diferencia sería una columna de ceros. No es una
magnitud continua sino un **descenso escalonado** de 38 a 0: cuenta jornadas, y su
histograma es por tanto casi uniforme con peines. `safe_from_drop_diff` no es una
magnitud sino la resta de dos indicadores, así que su forma son **tres barras**: 0 en
el 86 % de los partidos —o ninguno de los dos está matemáticamente salvado, o lo están
ambos— y ±1 en el resto, repartido casi por igual (215 y 199). Esa asimetría del
reparto frente a la masa del cero es lo que justifica leerla como indicador y no como
distancia. Las dos formas conviene leerlas junto a su imputación: el hueco de la
jornada 1 de este bloque **no se estima, se conoce** —38 partidos por jugar y nadie
salvado antes de disputarse un solo encuentro— y por eso D.2 lo trata con una
estrategia propia, `known`, cuyo relleno no mira la muestra.


In [ ]:
# A.5 . familia CLASIFICACION
plot_familia("clasificacion")


**H2H.** Las tasas por equipo se sesgan a la derecha (`home_h2h_draw_rate_ewm` 1.02) y sus diferencias se centran en 0 (`h2h_win_rate_ewm_diff` 0.01). El diferencial de goles del historial (`h2h_gd_ewm_diff`) es simétrico con cola moderada.

In [ ]:
# A.5 . familia H2H
plot_familia("h2h")


**Fatiga.** `home_fat_rest_days` tiene **cola derecha extrema** (skew 5.55): los parones de verano son descansos de ~100 días, no errores de captura. Los partidos jugados en los últimos 7 y 14 días son casi simétricos y de rango corto.

In [ ]:
# A.5 . familia FATIGA
plot_familia("fatiga")


**Bajas.** El bloque se ha reducido a cuatro lecturas y las cuatro dicen algo
distinto. El **recuento** total (`home_n_missing`) tiene sesgo derecho moderado; de él
se han retirado los desgloses por motivo —lesión, duda, sanción—, que sumaban el mismo
número repartido en cuatro columnas y cuyo detalle más raro, la sanción, era además un
evento casi siempre nulo. Las dos lecturas que **pesan** la ausencia en vez de contarla
son las que aportan lo que el recuento no puede: `home_out_minshare` mide qué porción de
los minutos del once falta y `home_out_minshare_max` el tamaño de la peor baja, de modo
que distinguen perder a tres suplentes de perder al delantero titular. Y
`home_out_value_pct` pondera lo mismo en euros: arrastra cola pesada, porque de vez en
cuando cae una estrella.

In [ ]:
# A.5 . familia BAJAS
plot_familia("bajas")


**Psicológicas.** Del bloque quedan tres columnas. La efectividad de definición
sobresale poco y con cola derecha, y `relative_perf` —el rendimiento frente a lo que el
propio historial hacía esperar— se acerca a la simetría. `home_post_big_win` es un
indicador raro que funciona como bandera: prácticamente siempre 0.

Su gemelo `post_big_fav_loss` —haber perdido siendo gran favorito— ya no se construye, y
el motivo merece anotarse porque es el único caso del proyecto en que una columna se
descarta por una medida empírica y no por un argumento de definición: su información
mutua con el resultado es del orden de 0.001–0.004, un orden de magnitud por debajo de la
feature útil más débil que se conserva, es decir, del ruido de muestreo. Descartar por
señal medida es la regla que más fácilmente se convierte en fuga si se aplica sobre el
bloque de evaluación, así que la medición se hizo sobre *dev* y la decisión quedó
escrita en `catalogo.py`, donde se lee y se audita, en vez de repetirse en cada ejecución
sobre los datos que toque.

In [ ]:
# A.5 . familia PSICO
plot_familia("psico")


**Contexto.** Las columnas numéricas del bloque describen el encuentro y no a
los equipos, y por eso ninguna tiene gemelo local/visitante. Su forma la imponen el
calendario y el mapa, no un fenómeno deportivo, y conviene leerla en esa clave.
`is_weekend` e `is_derby` son binarias y muy desiguales: la primera domina, porque el
grueso de cada jornada se disputa en sábado y domingo, y la segunda es rara, porque exige
que los dos clubes compartan comunidad autónoma. `away_travel_km` arrastra **cola derecha
estructural**: la masa se concentra en desplazamientos peninsulares de unos cientos de
kilómetros, hay un mínimo próximo a cero en los derbis de una misma ciudad y un extremo
aislado y perfectamente identificable en los viajes de ida y vuelta a Canarias, por
encima del millar de kilómetros. Esa cola no es un artefacto de captura sino la
geografía de la competición, así que ni se recorta ni se marca como anomalía: si su
asimetría lo justifica, la simetrización de D.4 la doblará como a cualquier otra.

La hora exacta de saque **ya no se construye**, y el bloque conserva solo la franja
categórica `kickoff_slot`, que se retrata en A.6 junto a las otras dos ordinales. La
razón está en el propio histograma que aquí se dibujaba: la hora era **multimodal**,
acumulaba masa en las franjas de emisión y dejaba valles entre ellas, de modo que
retrataba un horario televisivo y no una campana. El efecto que se hipotetiza es de
escalón —se juega a mediodía, por la tarde o de noche— y no de pendiente, así que la
franja es esa misma información ya agrupada en las tres categorías que significan algo, y
mantener las dos era ofrecer dos parametrizaciones del mismo atributo.

In [ ]:
# A.5 . familia CONTEXTO
plot_familia("contexto")


**Fuerza latente.** Las dos lecturas del bloque tienen formas deliberadamente
distintas y conviene leerlas juntas. Los dos parámetros latentes —`strength_attack` y
`strength_defence`— viven en **escala logarítmica y centrados en cero** por la
normalización a media geométrica 1 que impone la identificabilidad del modelo: su
histograma es aproximadamente simétrico y su recorrido, estrecho, porque el
encogimiento hacia la media de la liga comprime a los equipos con poca muestra hacia el
centro. Esa compresión es una propiedad del estimador, no una carencia del dato, y no
debe leerse como falta de discriminación: la cola del ataque llega a $\pm 0{,}9$, que
en la escala original equivale a marcar entre la mitad y el doble de lo que
correspondería a un equipo medio. El **conteo esperado de goles** que el ajuste produce —positivo, acotado por abajo y
con cola derecha suave— ya no viaja como columna: es exactamente lo que la terna de
probabilidades resume, y entregar las dos cosas era ofrecer el mismo pronóstico dos
veces. Sigue calculándose dentro del modelo, porque es el parámetro de la Poisson del
que salen las probabilidades; lo que no hace es llegar a la matriz. Los contrastes
`*_diff` de los dos parámetros latentes recuperan la simetría en torno a cero. Las probabilidades, por último,
están **acotadas en $[0,1]$** y `strength_p_draw` ocupa un rango notablemente estrecho:
el empate rara vez supera el 40 % bajo un modelo de Poisson, porque exige coincidencia
exacta de dos marcadores, y esa concentración es la razón por la que estas tres
columnas quedan **excluidas de la simetrización de D.4** —doblar una probabilidad la
sacaría de su escala natural sin ganar nada—.


In [ ]:
# A.5 . familia FUERZA
plot_familia("fuerza")


### A.6 · Univariante categórico
El conjunto tiene **once variables categóricas**, en dos formatos: cinco categóricas
de texto **ordinales** (`playstyle` y `position_zone`, cada una en su lectura local y
visitante, y `kickoff_slot`, que es del encuentro y por eso no se desdobla) y seis
**flags booleanos** de fatiga (`is_midweek`, `euro_trip_recent`, `is_post_intl_break`,
local/visitante). Las otras dos binarias del contexto —`is_weekend` e `is_derby`— no
comparecen aquí: el módulo las declara numéricas y viajan como 0/1, de modo que ya se
han retratado en A.5. La tabla siguiente resume cardinalidad, %NaN, moda y % de la moda
de las once; cada bloque muestra después sus frecuencias. Las de texto se codifican
**ordinales** (no one-hot): descenso < mid_table < europa < champions, defensivo <
equilibrado < ofensivo y mediodía < tarde < noche; los flags entran directamente
como 0/1.

In [ ]:
# --- A.6 . Univariante categorico: resumen + utilidades de dibujo -----------
# categoricas de texto (dtype "string") + flags booleanos (dtype bool). Se excluye el
# target y (A.4) y la meta. Los bool son categoricas binarias y tambien se retratan.
cat_txt  = [c for c in dev.columns
            if str(dev[c].dtype).startswith("string") and c not in META and c != "y"]
cat_bool = [c for c in dev.columns if dev[c].dtype == bool and c not in META]
cats = cat_txt + cat_bool
print(f"categoricas de texto ({len(cat_txt)}):", cat_txt)
print(f"flags booleanos  ({len(cat_bool)}):", cat_bool)

rows = []
for c in cats:
    s  = dev[c]; vc = s.value_counts(dropna=True)
    rows.append((c, s.nunique(), round(100*s.isna().mean(), 1),
                 str(vc.index[0]), round(100*vc.iloc[0]/len(s), 1)))
res = pd.DataFrame(rows, columns=["variable","card","%NaN","moda","%moda"])
print("\nresumen de las", len(cats), "categoricas (dev):")
print(res.to_string(index=False))

# El orden declarado aqui es el MISMO que preprocess congela en fit (ZONE_ORDER,
# PLAYSTYLE_ORDER y SLOT_ORDER): se repite en literal porque esta celda pertenece a
# la parte descriptiva y todavia no ha importado el modulo, y D.3 lo imprime desde
# el estado ajustado para que ambas lecturas se puedan contrastar.
ORD = {"playstyle":     ["defensivo","equilibrado","ofensivo"],
       "position_zone": ["descenso","mid_table","europa","champions"],
       "kickoff_slot":  ["mediodia","tarde","noche"]}
COL = {"defensivo":"#c62828","equilibrado":"#9e9e9e","ofensivo":"#2e7d32",
       "descenso":"#c62828","mid_table":"#9e9e9e","europa":"#1565c0","champions":"#6a1b9a",
       "mediodia":"#f9a825","tarde":"#1565c0","noche":"#283593"}

def barras_ordinal(cols, key, titulo):
    """Frecuencias (% de partidos) de una categorica de texto ordinal, local vs visitante."""
    order = ORD[key]
    fig, axes = plt.subplots(1, len(cols), figsize=(6*len(cols), 4))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, cols):
        pct    = dev[c].value_counts(dropna=False) / len(dev) * 100
        hasnan = dev[c].isna().any()
        labels = order + (["NaN"] if hasnan else [])
        vals   = [pct.get(l, 0) for l in order] + ([pct.get(np.nan, 0)] if hasnan else [])
        colr   = [COL.get(l, "#bbbbbb") for l in order] + (["#e0e0e0"] if hasnan else [])
        bars = ax.bar(range(len(labels)), vals, color=colr)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels([f"{l}\n({i})" if l in order else l for i, l in enumerate(labels)], fontsize=8)
        for r, v in zip(bars, vals):
            ax.text(r.get_x()+r.get_width()/2, v+0.5, f"{v:.1f}%", ha="center", fontsize=7)
        ax.set_title(f"{c}  (NaN {100*dev[c].isna().mean():.1f}%)", fontsize=9)
        ax.set_ylabel("% partidos")
    fig.suptitle(titulo, fontsize=11)
    plt.tight_layout(); plt.show()

def barras_flags(cols, titulo):
    """Tasa de True/False (% de partidos) de los flags booleanos."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 6)); axes = axes.ravel()
    for ax, c in zip(axes, cols):
        fa = 100*dev[c].eq(False).mean(); tr = 100*dev[c].eq(True).mean()
        bars = ax.bar([0, 1], [fa, tr], color=["#9e9e9e","#1565c0"])
        ax.set_xticks([0, 1]); ax.set_xticklabels(["False (0)","True (1)"], fontsize=8)
        for r, v in zip(bars, [fa, tr]):
            ax.text(r.get_x()+r.get_width()/2, v+1, f"{v:.1f}%", ha="center", fontsize=7)
        ax.set_title(f"{c}  (NaN {100*dev[c].isna().mean():.1f}%)", fontsize=9)
        ax.set_ylabel("% partidos"); ax.set_ylim(0, 100)
    for ax in axes[len(cols):]: ax.axis("off")
    fig.suptitle(titulo, fontsize=11)
    plt.tight_layout(); plt.show()


**Estilo de juego (`playstyle`).** Domina **equilibrado** (~62.6 % local, 62.5 %
visitante) sobre defensivo (~19 %) y ofensivo (~18 %). Las distribuciones local y
visitante son casi idénticas, lo esperable porque cada equipo aparece en ambos roles a
lo largo de la temporada. La etiqueta se asigna por rendimiento reciente, así que oscila
algo entre campañas (el peso de 'equilibrado' va de ~55 % en 2016–2017 a ~66–69 % en
2018–2019) sin alterar el orden. Los NaN (~0.5 %) son cold-start de las primeras
jornadas.

In [ ]:
# A.6 . bloque PLAYSTYLE
barras_ordinal(["home_playstyle","away_playstyle"], "playstyle",
               "A.6 - Estilo de juego (local vs visitante)")


**Zona de clasificación (`position_zone`).** `mid_table` reúne ~54 % de los partidos,
champions ~18 %, descenso ~15–16 % y europa ~9.5 %. Estas proporciones no son un hallazgo
empírico sino **estructurales**: reflejan el número de plazas de cada tramo sobre 20
equipos (11 de media tabla, 4 de Champions, 3 de descenso, 2 de Europa). Local y visitante
coinciden. Los NaN (~2.7 %) son de nuevo cold-start de la jornada 1, antes de que exista
clasificación.

In [ ]:
# A.6 . bloque POSITION_ZONE
barras_ordinal(["home_position_zone","away_position_zone"], "position_zone",
               "A.6 - Zona de clasificacion (local vs visitante)")


**Franja de emisión (`kickoff_slot`).** Es la única categórica del encuentro: no se
desdobla en local y visitante porque ambos equipos juegan, necesariamente, a la misma
hora. Reparte los partidos entre las tres franjas del reloj —mediodía por debajo de las
15 h, tarde entre las 15 y las 19, noche a partir de las 19— y su distribución retrata
el reparto audiovisual de la competición antes que ningún fenómeno deportivo. Entra
codificada como ordinal en ese mismo orden, que es el de la hora y no una jerarquía
inventada. Sus NaN, si aparecen, señalan partidos sin sello de saque en el máster: un
hueco de **cobertura de la fuente**, no un arranque de temporada, y la propia figura los
dibuja como categoría aparte para que no queden absorbidos por la franja modal.

In [ ]:
# A.6 . bloque KICKOFF_SLOT
barras_ordinal(["kickoff_slot"], "kickoff_slot",
               "A.6 - Franja de emision (unica: es del encuentro, no de un equipo)")


**Flags de fatiga (booleanos).** Tres indicadores binarios por equipo, **sin NaN**.
Son eventos minoritarios: `is_midweek` (partido entre semana) marca **True en ~13 %** de
los casos, `is_post_intl_break` (vuelta de parón FIFA) en ~8.5 % y `euro_trip_recent`
(viaje europeo reciente) en ~4.6 %, el más raro, lógico porque solo afecta a los equipos
en competición europea. Local y visitante comparten tasa de partido entre semana y de
parón (es el mismo encuentro) y difieren muy poco en viaje europeo.

In [ ]:
# A.6 . bloque FLAGS DE FATIGA (booleanos)
barras_flags(cat_bool, "A.6 - Flags de fatiga (tasa de True, %)")


---
## Parte B — Análisis bivariante y señal

¿Qué variables *separan* H/D/A? Antes de preprocesar conviene saber qué lleva
señal, para no tirar por error algo valioso y para tener un baseline honesto.

### B.1 · El mercado como baseline (el número a batir)
La probabilidad implícita des-margen del mercado es un modelo durísimo y fija la
referencia contra la que se juzga todo el TFM. En dev el **consenso** logra **log-loss
0.968 · accuracy 0.537**, con un **overround medio del 4.7 %** y una curva de calibración
que pega la diagonal (bien calibrado, sin sesgo sistemático; solo *infra*-confía un poco
en los locales muy favoritos —decil alto: implica 0.77, se cumple 0.80—).

**El precio admite tres lecturas y no es evidente a priori cuál es la exigente.** El
**consenso** es la media de las casas recogidas. **Pinnacle** opera con margen estrecho y
admite apostantes informados en lugar de limitarlos, por lo que su cotización suele estar
mejor calibrada que la media del sector. **Bet365** es la casa minorista de gran difusión:
la referencia que cualquier lector reconoce y con la que compararía el pronóstico sin
ayuda. Batir un consenso peor que el mejor precio disponible no demostraría nada, así que
se miden **las tres** y el número a batir que se declara es el **mejor de ellas**. La
comparación se hace sobre la **intersección** de partidos evaluables por los tres precios:
si cada uno se puntuara sobre su propia cobertura, la diferencia mezclaría calidad del
precio con dificultad de la muestra y dejaría de ser interpretable. `baselines_report`
implementa exactamente eso y devuelve la referencia elegida.

**De las tres, solo el consenso es una variable del modelo**, y la asimetría es
deliberada. Las tres ternas des-margen describen el mismo suceso con correlaciones muy
por encima de 0.95: meterlas todas obligaría a la poda a romper el empate de forma
arbitraria y a publicar un modelo cuya columna de precio depende de qué casa cubriera
mejor qué temporada. Así que el catálogo declara **una sola** terna de probabilidad
implícita —`mkt_p_home/draw/away`, el consenso— y lo que se conserva de las otras dos es
lo que **no** es redundante: de Pinnacle, la **divergencia** frente al consenso
(`sharp_div_*`), que mide desacuerdo entre precios y no nivel de precio; de Bet365, nada,
porque el consenso ya la promedia.

Las cuotas crudas de las tres casas —`odds_ps_*`, `odds_b365_*`, `odds_avg_*` y su
cierre— **siguen viajando en el artefacto** como columnas de servicio, fuera de
`feature_cols`. De ahí las reconstruye esta celda: la terna de Pinnacle sale de
`odds_ps_*` y la de Bet365 de `odds_b365_*`, ambas con la **misma** función de
des-margen que el consenso, para que la diferencia entre los tres baselines mida el
precio y no el método. Lo mismo hace `mkt_p_b365_*`, que se calcula en la fase de
features y viaja etiquetada como servicio precisamente para que la evaluación pueda
usarla sin que el modelo la vea.

Puesto en contexto, esto dice dos cosas. Primero, el mercado **bate con holgura a los
baselines tontos**: "predecir siempre local" acierta 45.1 % (la clase mayoritaria) y el
prior fijo de clases da log-loss 1.068; el mercado sube a 53.7 % y baja a 0.968.
Segundo —y más revelador— el mercado **casi nunca apuesta por el empate**: lo elige como
resultado más probable solo **7 de 3420 veces (0.2 %)** y su probabilidad implícita de
empate **nunca supera 0.36**, aunque los empates son el **26.4 %** de los partidos. El
empate es un punto ciego estructural del 1X2: es frecuente pero casi nunca es la apuesta
más probable de nadie. Por eso el accuracy engaña (un modelo no "canta" empates, reparte
probabilidad) y **el log-loss es la vara honesta a batir**.

In [ ]:
# --- B.1 · El mercado como baseline (el número a batir) --------------------
# El precio admite tres lecturas: el CONSENSO de las casas recogidas (mkt_p_*), el
# precio de Pinnacle, de margen estrecho, y el de Bet365, la casa minorista de gran
# difusión. Se miden las tres y el número a batir es el MEJOR, comparado sobre la
# intersección de filas evaluables por todas. Se mide SOLO en dev.
#
# De las tres, solo el consenso es VARIABLE. Las otras dos son referencia de
# evaluación y por eso ya no viajan como probabilidad: la de Bet365 se construye en
# `columns.SERVICIO_DERIVADO` y la de Pinnacle la reconstruye `sharp_baseline` de sus
# cuotas crudas `odds_ps_*`, que sí viajan como servicio porque el backtest liquida
# con ellas. Las tres se des-marginan con la MISMA función, de modo que la diferencia
# entre ellas mide el precio y no el método.
from sklearn.metrics import log_loss
import preprocess as pp

rep = pp.baselines_report(dev)
for nombre, b in rep["precios"].items():
    if b is None:
        print(f"baseline {nombre:9s}: no evaluable - {rep['note']}"); continue
    print(f"baseline {nombre:9s}: log-loss = {b['log_loss']:.4f}   accuracy = {b['accuracy']:.3f}"
          f"   (n={b['n']}, cobertura {b['coverage']:.1%})")
if rep["paired"]:
    par = rep["paired"]
    detalle = "  ".join(f"{n} {v['log_loss']:.4f}" for n, v in par["por_precio"].items())
    deltas  = "  ".join(f"{n} {d:+.4f}" for n, d in par["delta_log_loss"].items()
                        if n != "consenso")
    print(f"comparacion pareada sobre la interseccion (n={par['n']}, "
          f"cobertura {par['coverage']:.1%}): {detalle}")
    print(f"delta de log-loss frente al consenso: {deltas}")
elif rep["note"]:
    print("comparacion pareada no concluyente:", rep["note"])
print(f"-> NUMERO A BATIR = baseline '{rep['reference']}'")

# contexto del consenso: overround, baselines tontos y el punto ciego del empate
P = pp.market_baseline(dev)
mask = P.notna().all(axis=1)
y  = dev.loc[mask, "y_num"].values          # 0=H(local) 1=D 2=A, alineado con mkt_p_*
Pv = P.loc[mask, pp.MARKET_PROB_COLS].values
ocols = ["odds_avg_h","odds_avg_d","odds_avg_a"]
over  = ((1/dev.loc[mask, ocols]).sum(1).mean() - 1) if set(ocols).issubset(dev.columns) else float("nan")
maj   = (y == 0).mean()                                   # "siempre local"
prior = np.bincount(y, minlength=3) / len(y)
ll_prior = log_loss(y, np.tile(prior, (len(y), 1)), labels=[0,1,2])
print(f"\noverround medio del consenso = {over*100:.2f} %")
print(f"referencias tontas -> siempre local: acc {maj:.3f} | prior fijo: log-loss {ll_prior:.4f}")

pred = Pv.argmax(1)
print(f"empate como favorito del mercado: {int((pred==1).sum())} de {len(y)} "
      f"({(pred==1).mean()*100:.1f} %) | P(empate) implicita max = {Pv[:,1].max():.3f} "
      f"| empates reales = {(y==1).mean()*100:.1f} %")

# calibracion del favorito local por decil de prob. implicita
d = pd.DataFrame({"p": P.loc[mask,"mkt_p_home"].values, "win": (y==0).astype(int)})
d["bin"] = pd.qcut(d["p"], 10, labels=False, duplicates="drop")
cal = d.groupby("bin").agg(p_pred=("p","mean"), p_obs=("win","mean"), n=("win","size"))
fig, ax = plt.subplots(figsize=(4.5,4.5))
ax.plot([0,1],[0,1],"--",color="grey")
ax.plot(cal["p_pred"], cal["p_obs"], "o-", color="#1565c0")
ax.set_xlabel("prob. implícita (local)"); ax.set_ylabel("frecuencia observada")
ax.set_title("Calibración del mercado - local (dev)")
plt.tight_layout(); plt.show()
print(cal.round(3).to_string())

### B.2 · Forma vs objetivo
Diferenciales de forma reciente del local (versión home−away) frente a `y`. Las medianas
caen de forma **monótona H > D > A** en los balances de gol y de peligro
—`form_points_ewm_diff`, `cum_xgd_pg_diff` y `form_deep_ewm_diff` (jugadas peligrosas,
la que más abre)—, con las cifras exactas en el gráfico. El solape entre cajas es grande:
**señal débil pero real** — aporta, no decide sola.

Ahora bien, **no toda la "forma" separa**, y conviene enseñarlo: `form_ppda_ewm_diff`
(intensidad de presión) es **no monótona** — presionar más que el rival no se traduce en
ganar. Es un recordatorio de que el bloque de forma es grande pero desigual: los balances
de gol/xG discriminan, los indicadores de estilo mucho menos, y esa diferencia importará
cuando la selección hacia delante del 05 tenga que decidir a quién deja entrar.

**Las lecturas de la memoria frente al mismo objetivo.** Cada atributo de forma se ofrece
en **dos** parametrizaciones —ventana rectangular de N partidos y decaimiento exponencial
de semivida declarada, ambas fijadas en `catalogo.FORMA_VENTANA` y
`catalogo.FORMA_SEMIVIDA`—, y junto a ellas viajan tres bloques que miden cosas
distintas: el **acumulado de temporada** (`cum_*_pg_diff`), la forma **restringida al
escenario** (`venue_form_*`, solo partidos en casa para el local y solo fuera para el
visitante) y la **dureza del calendario recorrido** (`form_opp_elo_z_*`, la fuerza media
--tipificada dentro de la jornada-- de los rivales ya enfrentados). La tabla que sigue al gráfico las enfrenta al objetivo con
una medida comparable entre escalas: la separación entre la mediana de H y la de A
expresada en desviaciones típicas de la propia columna, única forma de poner en el mismo
eje una diferencia de xG, un balance de puntos y una fuerza media de rival.

Cabe esperar que las dos parametrizaciones se parezcan: la exponencial es una media
ponderada del mismo historial que la ventana. Pero **parecerse no es coincidir**: la
exponencial pondera de forma continua, mientras que la ventana trata por igual el último
partido y el N-ésimo y descarta de golpe el siguiente. Los otros tres bloques miden por
construcción algo que ninguna media de rendimiento propio contiene: dos equipos con el
mismo balance de puntos no han hecho el mismo mérito si uno visitó a los tres primeros y
el otro recibió a los tres últimos, ni valen lo mismo esos puntos si uno los hizo en casa
y el otro fuera.

Nada de esto **selecciona**. Es la comprobación descriptiva de que las lecturas no son la
misma columna repetida y de que, por tanto, tiene sentido dejarlas competir en la poda de
D.3 y, a las que sobrevivan, en la selección hacia delante del 05.

In [ ]:
# --- B.2 · Forma vs objetivo -----------------------------------------------
# Señal esperada: a mayor diferencial de forma del local, más victorias locales
# (monotonía H > D > A). form_ppda_5_diff se incluye a proposito como CONTRAEJEMPLO:
# no todo lo que huele a "forma" separa el resultado.
cols = [c for c in ["form_points_ewm_diff","cum_xgd_pg_diff","form_deep_ewm_diff",
                    "form_ppda_ewm_diff"]
        if c in dev.columns]
fig, axes = plt.subplots(1, len(cols), figsize=(3.6*len(cols), 3.6))
if len(cols) == 1: axes = [axes]
for ax, c in zip(axes, cols):
    ax.boxplot([dev.loc[dev["y"]==r, c].dropna() for r in ["H","D","A"]],
               labels=["H","D","A"], showfliers=False)
    ax.axhline(0, color="grey", lw=.8, ls="--"); ax.set_title(c, fontsize=9)
plt.tight_layout(); plt.show()
for c in cols:
    med = dev.groupby("y")[c].median().reindex(["H","D","A"])
    mono = "monótona" if med["H"] > med["D"] > med["A"] else "NO monótona"
    print(f"{c:20s} mediana H/D/A: {med.round(3).values}  ({mono})")

# --- Las tres lecturas de la memoria, frente al mismo objetivo -------------
# La pregunta NO es cual gana -eso lo decide la regularizacion del 05, pliegue a
# pliegue- sino si las lecturas miran lo mismo: si la exponencial ordenase H/D/A
# igual y con la misma separacion que la ventana, la segunda parametrizacion seria
# redundante y no habria por que calcularla. Se compara con la separacion de
# medianas H-A normalizada por la sigma de cada columna, unica forma de poner en el
# mismo eje unos puntos por partido, un acumulado y un rating medio de rival.
#
# De las CINCO lecturas que llego a haber quedan estas: el catalogo conserva por
# atributo una ventana corta y la exponencial (regla 1), y de las que resumian la
# temporada entera solo el acumulado. La forma ajustada por Elo previo -el residuo
# S-E- ya no se construye: era una lectura de la SUERTE tanto como del merito, y esa
# parte no es reproducible (regla 5). Lo que si sobrevive del ajuste al rival es la
# dureza del calendario recorrido, que no mide desviacion sino contra quien se jugo.
import catalogo as cat
lecturas = [(f"ventana N={cat.FORMA_VENTANA:g}",   "form_points_3_diff"),
            (f"exponencial hl={cat.FORMA_SEMIVIDA:g}", "form_points_ewm_diff"),
            ("acumulado de temporada",             "cum_points_pg_diff"),
            (f"restringida al escenario (N={cat.FORMA_ESCENARIO_VENTANA:g})",
                                                   "venue_form_points_5_diff"),
            ("dureza del calendario (fuerza rival)", "form_opp_elo_z_ewm_diff")]
filas = []
for nombre, c in lecturas:
    if c not in dev.columns:
        continue
    med, sd = dev.groupby("y")[c].median().reindex(["H","D","A"]), dev[c].std()
    filas.append({"lectura": nombre, "columna": c,
                  "med H": med["H"], "med D": med["D"], "med A": med["A"],
                  "(H-A)/sigma": (med["H"] - med["A"]) / sd,
                  "monotona": bool(med["H"] > med["D"] > med["A"]),
                  "%NaN": dev[c].isna().mean() * 100})
print("\nseparacion H/D/A de cada lectura de la memoria (dev):")
print(pd.DataFrame(filas).round(3).to_string(index=False))

# Solape efectivo entre parametrizaciones del MISMO atributo: si fuese r~1 la
# exponencial seria una copia y sobraria; el valor real dice cuanta informacion
# distinta aporta, y es lo que la poda colineal de D.3 tendra que arbitrar.
par = [c for _, c in lecturas if c in dev.columns and "points" in c]
if len(par) > 1:
    print("\ncorrelacion entre parametrizaciones del mismo atributo (puntos, dev):")
    print(dev[par].corr().round(3).to_string())


### B.3 · Elo, valor de plantilla y clasificación vs objetivo
Priors de fuerza frente al resultado. El winrate local crece de forma casi lineal con la
diferencia de valor de plantilla y con la zona del local (**descenso 34 % → mid_table
40 % → europa 51 % → champions 68 %**).

Dos matices que solo aparecen al desglosar el resultado por zona. Primero, en la **zona
de descenso el local gana menos (34 %) de lo que gana su rival a domicilio (38 %)**:
para los peores anfitriones la ventaja de campo se da la vuelta. Segundo, **el empate es
casi plano entre zonas (19–28 %)**: ser un local fuerte compra victorias, no menos
empates — un equipo *champions* todavía empata el 19 % de sus partidos. Son priors
fuertes, aunque por debajo del mercado en MI.

**Dos de los tres ejes de esta celda se calculan aquí y no viajan como columna**, y
merece la pena decir por qué. Del valor de plantilla, el modelo recibe el **par**
(`home_squad_value`, `away_squad_value`); el log-ratio que ordena el gráfico es una
función determinista de ese par, así que el catálogo no lo declara y la celda lo calcula
al vuelo. Del Elo ocurre algo parecido con un matiz: lo que viaja en la matriz es el par
**tipificado** (`home_elo_z`, `away_elo_z`), mientras que la logística
`prematch_derived.elo_expected_score` --la única del proyecto, que **no produce ninguna
columna**-- está definida sobre una diferencia en puntos de rating. La celda recupera por
eso el nivel crudo del **máster**, donde se conserva justamente como columna descriptiva
y auditable, y dibuja la curva con él. Esa curva queda por
encima de la victoria pura porque mide puntos esperados —con medio-empate— y no
P(gana local); lo relevante es la monotonía, no el nivel.

La distinción no es de estilo. Una transformación monótona de una variable que ya está en
la matriz no añade información a ningún modelo, pero sí añade una columna que hay que
imputar, escalar, podar y explicar. Se calcula donde sirve —en el análisis, para leer el
prior en una escala interpretable— y no se arrastra hasta el modelo.

In [ ]:
# --- B.3 · Elo, valor de plantilla y clasificación vs objetivo -------------
# Los dos priores de fuerza -el economico y el de rating- y la zona de la tabla,
# frente al resultado. Deben dar un winrate local creciente, y el desglose H/D/A por
# zona ensena ademas como se comporta la ventaja de campo en cada tramo.
#
# Dos de las columnas que este bloque retrataba ya no viajan en la matriz, y ninguna
# de las dos por falta de interes: `squad_value_logratio` es una funcion del par de
# valores que si viaja, y `elo_exp_home` es una funcion determinista de la resta de
# ratings (r=1). El catalogo emite el par y deja fuera lo deducible (reglas 2 y 3).
# Que no viajen no impide retratarlas: se reconstruyen aqui al vuelo, con la misma
# formula que las producia, porque lo que este apartado mide es el PODER del prior
# -si ordena el resultado y si esta calibrado- y ese poder no cambia porque la
# transformacion se aplique dentro del modelo en vez de antes de entrar.
import numpy as np
import prematch_derived as pdrv

win = (dev["y"] == "H").astype(int)  # victoria local

if {"home_squad_value", "away_squad_value"}.issubset(dev.columns):
    logratio = np.log1p(dev["home_squad_value"]) - np.log1p(dev["away_squad_value"])
    dz = pd.DataFrame({"x": logratio, "win": win}).dropna()
    dz["dec"] = pd.qcut(dz["x"], 10, labels=False, duplicates="drop")
    print("winrate local por decil de log-razon de valor de plantilla (dev, %):")
    print((dz.groupby("dec")["win"].mean()*100).round(1).to_string())

if "home_position_zone" in dev:
    tab = pd.DataFrame({
        "local":     dev.assign(v=(dev["y"]=="H").astype(int)).groupby("home_position_zone")["v"].mean()*100,
        "empate":    dev.assign(v=(dev["y"]=="D").astype(int)).groupby("home_position_zone")["v"].mean()*100,
        "visitante": dev.assign(v=(dev["y"]=="A").astype(int)).groupby("home_position_zone")["v"].mean()*100,
    }).round(1)
    print("\nreparto H/D/A por zona de clasificación del local (%):")
    print(tab.to_string())

# El nivel crudo del Elo NO viaja en la matriz -lo que entra al modelo es el par
# tipificado dentro de la jornada- y la logistica de abajo esta definida sobre una
# diferencia en PUNTOS de rating. Se recupera por eso del master, donde el nivel se
# conserva como columna descriptiva y auditable, cruzando por `match_key`. Es la
# unica lectura de este notebook que sale del master y no del conjunto de variables,
# y se hace explicito para que no parezca que `dev` trae una columna que no trae.
_niv = (pd.read_parquet(PROC / "master_matches.parquet",
                        columns=["match_key", "home_elo", "away_elo"])
          .set_index("match_key")
          .reindex(dev["match_key"].to_numpy()))
print("\npartidos de dev con nivel de Elo en el master: "
      f"{int(_niv['home_elo'].notna().sum())} de {len(dev)}")

if _niv["home_elo"].notna().any():
    # Puntuacion esperada del local segun el Elo, con la misma ventaja de campo
    # (HOME_ADV_ELO) que emplea el resto del pipeline. Es PUNTUACION esperada -el
    # empate cuenta medio- y no P(gana local), asi que su curva queda por encima de
    # la diagonal por construccion: lo que se juzga aqui es la MONOTONIA.
    ediff = pd.Series((_niv["home_elo"] - _niv["away_elo"]).to_numpy(),
                      index=dev.index)
    de = pd.DataFrame({"p": pdrv.elo_expected_score(ediff), "win": win}).dropna()
    de["bin"] = pd.qcut(de["p"], 10, labels=False, duplicates="drop")
    ce = de.groupby("bin").agg(p=("p","mean"), obs=("win","mean"))
    fig, ax = plt.subplots(figsize=(4.5,4.5))
    ax.plot([0,1],[0,1],"--",color="grey")
    ax.plot(ce["p"], ce["obs"], "o-", color="#6a1b9a")
    ax.set_xlabel(f"puntuacion esperada del Elo (ventaja local {pdrv.HOME_ADV_ELO:g})")
    ax.set_ylabel("winrate local observado")
    ax.set_title("Calibración del Elo (dev)")
    plt.tight_layout(); plt.show()

if {"home_elo_z", "away_elo_z"}.issubset(dev.columns):
    # Y la misma monotonia sobre lo que de verdad llega al modelo: la diferencia de
    # fuerza TIPIFICADA. No hay logistica calibrada para esta escala -ni hace falta,
    # porque el estimador aprende la suya-, asi que se lee por deciles.
    dzz = pd.DataFrame({"x": dev["home_elo_z"] - dev["away_elo_z"],
                        "win": win}).dropna()
    dzz["dec"] = pd.qcut(dzz["x"], 10, labels=False, duplicates="drop")
    print("\nwinrate local por decil de diferencia de fuerza tipificada (dev, %):")
    print((dzz.groupby("dec")["win"].mean() * 100).round(1).to_string())


### B.4 · Poder discriminante por familia
Información mutua marginal de cada feature con el target (solo dev). El **TOP-15 lo
copan casi por completo las cuotas**; las **familias no-mercado que se cuelan son valor,
Elo y fuerza latente**, que quedan a un paso de la mejor cuota. Es decir, el prior de
fuerza fundamental **no está tan lejos** del mercado columna a columna; la ventaja del
mercado está en agregarlo todo en un solo precio.

Por familia la señal media ordena **mercado > elo > valor > clasificación > forma >
h2h**, con **fatiga / psico / bajas** casi en ruido. Destaca la **paradoja de la forma**:
es con diferencia la familia más grande —incluso después del recorte del catálogo— pero
su MI media es de las más bajas del grupo "útil" y **ninguna de sus columnas entra en el
TOP-15**. Conviene leer bien el dato: la MI marginal puntúa cada columna aislada, y lo
que se espera de la forma es justamente que aporte en agregación e interacción, no en
solitario. El desglose interno por lectura (segunda tabla) matiza la paradoja: la
ventana, la exponencial, el acumulado, el escenario y la dureza del calendario puntúan en
el mismo orden de magnitud, lo que descarta que alguna sea ruido, pero **no** demuestra
que aporten cosas distintas —la MI marginal ignora la redundancia entre columnas y
premiaría por igual a dos lecturas que dijeran lo mismo—. Eso lo resuelven la poda de
D.3 y, después, la selección hacia delante del 05.

La tabla incluye también el **contexto** —calendario y geografía—, y su lectura pide
dos cautelas más. La primera es aritmética: es una de las familias más pequeñas del
conjunto, de modo que su media se calcula sobre muy pocas columnas y basta una para
desplazarla. La segunda es conceptual: de la franja de saque o de los kilómetros
recorridos no se espera señal marginal —ninguna hora gana partidos— sino
**condicionamiento**, esto es, que modulen cómo se traduce el resto del vector en el
resultado. Una MI marginal, que puntúa cada columna aislada frente al target, es por
construcción ciega a ese papel.

La **fuerza latente** entra en esta tabla con uno de los mejores registros no-mercado del
conjunto: la terna 1X2 del modelo de goles puntúa por encima del par de Elo. La media de
la familia queda además por encima de la del mercado, aunque esa comparación es engañosa
y conviene desactivarla explícitamente: el bloque tiene pocas columnas y todas apuntan a
lo mismo, mientras que el de cuotas arrastra la cola de derivadas de estructura de precio
que rebaja su media. Lo que sí se puede afirmar es que el prior de fuerza **construido
desde el marcador** puntúa por encima del construido desde el rating, lo cual es
coherente con que el primero incorpore el volumen de goles y el segundo solo el balance
de victorias. Lo que **no** se puede afirmar es que aporte señal *nueva*: su correlación
con la diferencia de Elo ronda 0.94, así que ambas describen en buena medida el mismo eje
de calidad y la MI marginal, ciega a la redundancia, las premia por separado. La parte
verdaderamente propia del bloque es la **descomposición en ataque y defensa**
—`strength_attack_diff` y `strength_defence_diff`, que correlacionan poco entre sí—; el
volumen agregado de goles esperados no se declara en el catálogo justamente porque es la
parte que el Elo ya cubre.

Todo esto es MI **marginal y exploratoria**: no descarta nada —eso lo hace la selección
hacia delante del 05, sobre bloque separado— y una familia débil en solitario aún puede
aportar combinada.

In [ ]:
# --- B.4 · Poder discriminante por familia (MI, exploratorio) --------------
# Información mutua de cada feature con el target, SOLO dev. Exploratorio: NO
# selecciona nada (eso es 05 con CV temporal), solo ordena la señal por familia.
import warnings
from sklearn.feature_selection import mutual_info_classif
# las categóricas son dtype "string" (no "object"); filtrar por si preprocess
# conoce su orden ordinal, no por dtype -> así SÍ se codifican antes del MI.
cmaps = {c: pp._categ_order_for(c) for c in dev.columns if pp._categ_order_for(c)}
work = pp._encode_categoricals(dev, cmaps)
cand = [c for c in work.columns if c not in pp._NON_FEATURE]
Xn = work[cand].apply(pd.to_numeric, errors="coerce")
with warnings.catch_warnings():
    # columnas enteramente NaN (época: cierre pre-19, bajas 16/17) -> su mediana
    # es un "mean of empty slice"; el fillna(0.0) posterior las neutraliza.
    warnings.simplefilter("ignore", RuntimeWarning)
    X = Xn.fillna(Xn.median()).fillna(0.0)  # relleno solo para el MI exploratorio
mi = pd.Series(mutual_info_classif(X, dev["y_num"].values, random_state=0),
               index=cand).sort_values(ascending=False)
famcol = fam.reindex(mi.index)
print("TOP-15 features por MI (dev):")
print(mi.head(15).round(4).to_string())

# cuanto se acerca lo NO-mercado: mejor feature de fuera del mercado
print("\nmejor señal NO-mercado (features de fuera del bloque de cuotas):")
print(mi[famcol != "mercado"].head(5).round(4).to_string())

# MI media / máx / nº de columnas por familia -> muestra la paradoja de la forma
print("\nseñal por familia (media, máx, nº columnas):")
by = mi.groupby(famcol).agg(["mean","max","count"]).sort_values("mean", ascending=False)
print(by.round(4).to_string())

# Dentro de la forma: puntuan igual las tres lecturas? Descriptivo, no decisorio.
# La MI marginal ignora la redundancia, asi que este desglose sirve para RECHAZAR
# la hipotesis fuerte "alguna lectura es ruido", no para elegir entre ellas.
def _lectura(c):
    if "opp_elo" in c:                     return "forma . dureza del calendario"
    if c.startswith("venue_form"):         return "forma . restringida al escenario"
    if c.startswith("cum_"):               return "forma . acumulado de temporada"
    if c.endswith(("_ewm", "_ewm_diff")):  return "forma . exponencial"
    return "forma . ventana corta"

mi_f = mi[famcol == "forma"]
print("\nMI marginal dentro de la familia forma, por lectura de la memoria:")
print(mi_f.groupby(mi_f.index.map(_lectura))
          .agg(["mean", "max", "count"]).round(4).to_string())

---
## Parte C — Estructura del dataset (colinealidad, constantes, drift)

Problemas que no se ven mirando una variable a la vez: redundancia entre
columnas, columnas muertas y cambios de distribución en el tiempo.

### C.1 · Multicolinealidad
La curación de la capa de features ha vaciado de antemano dos de los tres tipos de
redundancia que este apartado encontraba, y merece la pena decir cuáles y por qué, porque
es la diferencia entre podar y no construir.

Las **simetrías del h2h** —la tasa de empate de un equipo es idéntica a la de su rival, de
modo que su diferencia era una columna de ceros— desaparecen porque el catálogo declara
esa tasa como magnitud del **enfrentamiento** y la emite una sola vez, sin rótulo de lado.
Los **duplicados de construcción** —`n_missing` contra el recuento alternativo de
ausentes, `matches_prev` del local contra el del visitante, el saque entre semana repetido
en los dos lados— desaparecen porque de cada atributo se emite una sola parametrización y,
cuando el número es el mismo para los dos equipos, un solo lado.

Lo que **sigue apareciendo** es el tercer tipo, y no puede ser de otro modo: las **cuotas
casi idénticas** entre casas y entre apertura y cierre (`odds_avg/max/ps/b365`). Esas
columnas no son variables —viajan como servicio para que el backtest liquide con ellas— y
por tanto ni entran en la poda ni compiten por un coeficiente.

La **poda greedy de D.3** (|r|>0.95) sigue en su sitio para lo que no se puede anticipar:
conserva de cada grupo el de mayor MI×cobertura y protege la terna `mkt_p_*`, que es a la
vez variable y línea base. Las ternas de Bet365 y de Pinnacle no necesitan protección
porque ni siquiera llegan a candidatas —quedan apartadas antes, en `columns.SERVICIO`, por
ser referencia de evaluación y no variable— y viajan igualmente en el artefacto para que
los otros dos baselines sigan siendo calculables.

In [ ]:
# --- C.1 · Multicolinealidad -----------------------------------------------
# Pares casi-perfectos: informan la poda greedy de D.3 (se queda el de mayor
# MI×cobertura). Aquí solo se listan; la decisión se ejecuta en preprocess.fit.
numf = work[cand].apply(pd.to_numeric, errors="coerce")
corr = numf.corr().abs()
up = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = (up.stack().reset_index()
           .rename(columns={"level_0":"a","level_1":"b",0:"r"}))
hi = pairs[pairs["r"] > 0.98].sort_values("r", ascending=False)
print(f"pares con |r|>0.98: {len(hi)}  (muestra)")
print(hi.head(25).round(3).to_string(index=False))

### C.2 · Columnas constantes / casi-constantes
El caso que este apartado identificó en su día —las dos diferencias de tasa de empate del
h2h, `h2h_draw_rate_*_diff`, que eran **todo-cero**— ya no puede volver a aparecer, y no
porque se pode sino porque no se construye: un empate lo es para los dos equipos a la vez,
así que su diferencia es cero en todas las filas por definición y no por casualidad
muestral. El catálogo declara esa tasa como magnitud del enfrentamiento y la emite una
sola vez.

La comprobación se mantiene de todos modos, y aquí sí como poda y no como declaración: si
una columna resulta constante en *dev* —el caso vivo es el bloque de calidad de ocasión
cuando el volcado de tiros no se ha ejecutado— eso solo lo dicen los datos, y `fit(dev)`
la retira antes de la imputación.

In [ ]:
# --- C.2 · Columnas constantes / casi-constantes ---------------------------
nun = numf.nunique(dropna=True)
const = nun[nun <= 1].index.tolist()
std = numf.std()
almost = std[(std < 1e-6) & (~std.index.isin(const))].index.tolist()
print("constantes (nunique<=1):", const)
print("casi-constantes (std<1e-6):", almost)

### C.3 · Drift temporal (cobertura y distribución en el tiempo)
**Cobertura**: dos huecos de época, no de captura — las cuotas de cierre saltan de
**0 % a 100 % en 19/20** y las bajas de **0 % a 100 % en 17/18**; el valor de plantilla
tiene cobertura completa desde 16/17. **Nivel**: el valor medio sube de **202 → 309 M€
(16/17→18/19)**, corrige con el COVID (**272 M€ en 19/20**) y vuelve a subir hasta
**~297 M€ (25/26)**. No se normaliza por temporada: el modelo usa **diferencias
home−away**, que absorben la inflación común; el hueco se trata con flag + imputación
por causa en D.2.

La lectura por columna testigo se acompaña de la **matriz de cobertura** publicada junto al
conjunto (`coverage_matrix.parquet`), que declara partido a partido si cada fuente llegó
entera. Es la versión que se cita en la memoria: no depende de que la columna elegida
represente bien a su bloque, y separa el hueco de época —la fuente aún no existía— del hueco
de captura —la fuente existía y no se recogió—, que es la distinción de la que depende que la
imputación de D.2 esté bien motivada.

In [ ]:
# --- C.3 · Drift temporal (cobertura y nivel en el tiempo) -----------------
# Familias con hueco de época (cuotas de cierre 16-19, bajas 16/17) e inflación
# del valor de plantilla. Justifica flag + imputación por causa (D.2).
t = feat["season_year"].map(temporada)
for col in ["odds_close_avg_h","home_out_value","home_squad_value"]:
    if col in feat:
        c = (feat.assign(t=t).groupby("t")[col]
                 .apply(lambda s: round(100*(1-s.isna().mean()))))
        print(f"cobertura {col} (%) por temporada:"); print(c.to_string()); print()
if "home_squad_value" in feat:
    print("valor medio de plantilla local por temporada (inflación esperada):")
    print(feat.assign(t=t).groupby("t")["home_squad_value"].mean().round(0).to_string())

# visual: hueco de cobertura + inflación del valor, para leerlo de un vistazo
cov_cols = {"cierre (odds_close_avg_h)":"odds_close_avg_h",
            "bajas (home_out_value)":"home_out_value",
            "valor (home_squad_value)":"home_squad_value"}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
ax = axes[0]
for lab, col in cov_cols.items():
    if col in feat:
        cov = feat.assign(t=t).groupby("t")[col].apply(lambda s: 100*(1-s.isna().mean()))
        ax.plot(cov.index, cov.values, "o-", label=lab)
ax.set_ylabel("% cobertura"); ax.set_ylim(-5, 105)
ax.set_title("Cobertura por temporada (hueco de época)")
ax.legend(fontsize=7); ax.tick_params(axis="x", rotation=45)
ax2 = axes[1]
val = feat.assign(t=t).groupby("t")["home_squad_value"].mean()
ax2.plot(val.index, val.values, "s-", color="#2e7d32")
ax2.set_title("Valor medio de plantilla local (inflación)")
ax2.set_ylabel("M€"); ax2.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

# La lectura anterior toma tres columnas como testigo de sus bloques. La matriz de
# cobertura publicada permite hacer la misma lectura SIN elegir testigo: declara,
# partido a partido, si cada fuente llego entera. Es la version que va a la memoria,
# porque no depende de que la columna elegida represente bien a su bloque.
_pt = coverage_by_season(cov)
_fu = [c for c in cov.columns if c.startswith("fuente_")]
print("cobertura de fuente por temporada (proporcion de partidos completos):")
print(_pt.assign(season=_pt.season.map(lambda s: s if isinstance(s, str) else temporada(s)))
         [["season", "partidos"] + _fu].round(2).to_string(index=False))

print("\nprimera temporada con dato de cada fuente:")
for _c in _fu:
    _act = _pt.loc[_pt[_c] > 0, "season"]
    print(f"  {_c[7:]:20s}: {_act.iloc[0] if len(_act) else '—'}")

### C.4 · Re-verificación de leak-safety a nivel dataset
Recordatorio ejecutable de que el set sigue pasando los once invariantes duros de
`qa_features.run_qa` **antes** de preprocesar (por si el parquet en disco se
regeneró). Sirve de ancla: lo que salga del preprocesado también deberá pasar QA.

In [ ]:
# --- C.4 · Re-verificacion de leak-safety a nivel dataset ------------------
# El conjunto ya paso qa_features al construirse; se re-ejecuta aqui sobre el
# parquet que este notebook ha leido, que puede no ser el mismo si se regenero
# entre medias. No se envuelve en try/except a proposito: un invariante duro que
# no puede detener el notebook deja de ser un invariante y pasa a ser un adorno.
from qa_features import run_qa

run_qa(feat, verbose=True)

---
## Parte D — Decisiones y preprocesado reproducible

Aquí se convierten los hallazgos de A–C en **decisiones registradas** y se
implementan en el módulo `scripts/features/preprocess.py`
(`fit(dev) → PreprocessState` / `apply(df, state)`), que el orquestador
`scripts/features/build_preprocess.py` ejecuta y persiste.

### Régimen de información — qué columnas pueden entrar en el ajuste
Ser **pre-partido** y ser **conocible en el momento de predecir** son propiedades
distintas. Toda variable de la cadena es estrictamente anterior al pitido inicial
—eso lo garantizan los invariantes antifuga de `qa_features`—, pero la predicción de
una jornada se emite con días de antelación y hay información pre-partido que en ese
instante todavía no se ha publicado. El caso que obliga a separar los dos planos es
el **precio de cierre**: las casas lo fijan minutos antes del saque, cuando la
jornada ya está puntuada.

Por eso el preprocesado se ajusta bajo dos **regímenes de información**, declarados
en `scripts/features/columns.py`:

| régimen | contenido | papel |
|---|---|---|
| `semanal` | solo lo conocible al emitir la predicción | es el modelo que se **publica**; su métrica es la que se defiende como capacidad predictiva del sistema |
| `retrospectivo` | añade el bloque de cierre | no se publica; su comparación con el semanal mide **cuánta información aporta el mercado entre la apertura y el cierre**, es decir, el techo alcanzable con información pre-partido completa |

**El régimen se aplica antes del ajuste, no después.** La razón está en la poda
colineal (D.3): es supervisada y compara pares, y la cuota de apertura correlaciona
con su gemela de cierre por encima de cualquier umbral razonable. Ajustar con las
dos presentes y filtrar el cierre a posteriori dejaría al régimen semanal **sin
ninguna de las dos**. Por el mismo motivo se filtran también los flags de faltante:
`flag_no_closing_odds` valdría casi siempre 0 en entrenamiento —solo 16/17–18/19
carece de cierre— y siempre 1 en inferencia, de modo que el modelo publicado leería
cada semana un valor que jamás vio al ajustarse.

El bloque de cierre se identifica por un **convenio de nombres**: toda columna
derivada del cierre lleva el token `close`. El convenio hace el predicado completo
*por construcción*, así que una variable nueva de movimiento de precio queda fuera
del régimen semanal sin editar ninguna lista; `scripts/features/test_regimen.py`
fija esa propiedad con datos sintéticos. Ambos regímenes se ajustan con **el mismo
protocolo**, que es la condición para que la diferencia entre ellos se lea como una
magnitud del dominio y no como el efecto de haber cambiado a la vez las variables y
el método.

Los dos ejes —fuga y momento de disponibilidad— son **independientes**: una columna
de cierre no es fuga (es pre-partido) y una columna de fuga lo es en los dos
regímenes.

In [ ]:
# --- Régimen de información · partición del conjunto de candidatas ---------
# El régimen decide qué columnas PUEDEN entrar en fit(), no cuáles sobreviven a la
# poda. Se declara en columns.py y queda impreso en el estado congelado.
import importlib
import columns as C; importlib.reload(C); importlib.reload(pp)

cand_ret = pp.feature_candidates(dev, "retrospectivo")
cand_sem = pp.feature_candidates(dev, "semanal")
fuera    = C.weekly_unavailable(cand_ret)

print(f"candidatas a feature   retrospectivo = {len(cand_ret)}   semanal = {len(cand_sem)}")
print(f"\nno conocibles al emitir la predicción ({len(fuera)}), token '{C._CLOSING_TOKEN}':")
for c in fuera:
    print(f"    {c}")
print("\npartición sin solape ni pérdida:",
      set(cand_sem) | set(fuera) == set(cand_ret) and not (set(cand_sem) & set(fuera)))
print("la apertura y la probabilidad implícita permanecen en el semanal:",
      [c for c in cand_sem if c.startswith("odds_avg_") or c.startswith("mkt_p_")])
print("la estructura del precio permanece en el semanal:",
      [c for c in cand_sem if c.startswith(("mkt_overround_","mkt_spread_","sharp_div_"))])

### D.0 · Decisión cerrada — lag t−1 del valor de plantilla (Transfermarkt)
**Decisión tomada: opción (a), laggear una temporada.** El prior económico de un
partido de la temporada `s` usa el valor de plantilla de `s−1` (ya cerrado), nunca
el de la temporada en curso. Se implementa en la **capa de consolidación**
(`scripts/consolidate/squadvalue.py`, `val["season"] += lag`, `lag=1`), no en el
preprocesado: así el lag forma parte de la *definición* de la feature y es
leak-safe por construcción (no depende de ninguna estadística del split).

**Por qué.** La página `saison_id/{year}` de Transfermarkt refleja el valor
*dentro* de esa temporada (mercado de invierno + revalorizaciones que ya
incorporan el rendimiento en curso); para la temporada vigente, además, el
`update` semanal lo re-scrapea en vivo. Usar ese valor para predecir un partido de
la misma temporada es *lookahead*. Con t−1 el prior es estrictamente pre-partido.

**Consecuencia y su tratamiento.** Los recién ascendidos no tienen valor de `s−1`
en Primera → `NaN` estructural (cold-start). No se rellena a ciegas: **D.2** lo
trata con `weak_prior` (cuantil bajo de dev) + `flag_no_squadvalue`, porque un
ascendido no es un equipo promedio sino de cola inferior — el mismo criterio con el
que un ascendido, sin posición previa en Primera, se ancla en la cola de la tabla. La
celda de código comprueba el patrón: el `NaN` se concentra en los ascendidos.

In [ ]:
# --- D.0 · Efecto del lag t-1 del valor de plantilla -----------------------
# Decisión: (a) laggear a t-1, implementado en scripts/consolidate/squadvalue.py
# (val.season += 1). Consecuencia esperada: los ascendidos NO tienen valor de s-1
# en Primera -> NaN estructural (cold-start), tratado por weak_prior en D.2.
sv = feat.assign(t=feat["season_year"].map(temporada))
print("% NaN de home_squad_value por temporada (0 = parquet aún PRE-lag):")
print(sv.groupby("t")["home_squad_value"]
        .apply(lambda s: round(100*s.isna().mean(),1)).to_string())
n_na = int(feat["home_squad_value"].isna().sum())
print(f"\ntotal partidos con valor local ausente: {n_na}")
print("→ si es 0, el parquet es PRE-lag; al regenerar con squadvalue.py laggeado")
print("  estos NaN aparecen en ascendidos y los absorbe weak_prior (D.2).")

### D.1 · Registro de decisiones (tabla maestra)
Una tabla explícita: por cada columna/grupo → **acción** (mantener / tirar / imputar /
transformar / régimen) + **motivo** + **parámetro** (constante de imputación, etc.).
Este registro es la auditoría del preprocesado y va tal cual a la memoria del TFM.
Las decisiones estructurales:
- **Régimen**: el bloque de cierre queda fuera del ajuste semanal por no existir
  cuando se emite la predicción; solo el retrospectivo lo incorpora.
- **Tirar** las 2 constantes h2h todo-cero (C.2).
- **Tirar** una columna de cada par colineal `|r|>0.95` (C.1), conservando la de mayor
  `MI×cobertura`.
- **Imputación por causa** (D.2), no un `fillna(0)` global a ciegas.
- **Ampliar antes de podar**: la forma se ofrece en sus tres lecturas —ventana,
  decaimiento exponencial y ajuste por fuerza del rival— sin decidir de antemano cuál
  sobrevive. La dimensión del conjunto candidato crece a propósito, porque el criterio
  que resuelve el solape es la poda supervisada por `MI×cobertura` y no una preferencia
  declarada por el analista.

In [ ]:
# --- D.1 · Registro de decisiones (tabla maestra) --------------------------
decisiones = pd.DataFrame([
 ("valor","home/away_squad_value (ascendidos)","imputar","cold-start t-1: un ascendido no es promedio, cae en la cola inferior","weak_prior q=0.10 dev + flag_no_squadvalue"),
 ("valor","squad_value_diff / logratio","no construir","dos reparametrizaciones del par que ya viaja: la resta y la log-razon se deducen de el","catalogo.PLANTILLA (regla 2)"),
 ("mercado","odds_close_* / close_move_*","régimen","no existen al emitir la predicción: solo entran en el retrospectivo","weekly_unavailable, token 'close'"),
 ("mercado","odds_close_* 16-19 (retrospectivo)","imputar","hueco de época (cierre no recogido), no ausencia real","mediana dev + flag_no_closing_odds"),
 ("bajas","home/away_out_* 16/17","imputar","fuente no recogida esa temporada","mediana dev + flag_no_absences"),
 ("bajas","n_injury / n_doubtful / n_suspension","no construir","desglose de `n_missing`, que es la lectura que se conserva; a el se suman las dos que PESAN la baja","catalogo.BAJAS_ATRIBUTOS (regla 4)"),
 ("h2h","h2h_* sin enfrentamiento","imputar","ausencia REAL: sin historial -> neutro 0","0 + flag_no_h2h"),
 ("h2h","h2h_draw_rate_ewm","sin lado","un empate lo es para los dos a la vez: su diferencia seria constante cero","catalogo.H2H_PARTIDO"),
 ("h2h","ventana rectangular de 8 enfrentamientos","no construir","misma pregunta con otro supuesto de olvido, y con uno o dos partidos por temporada apenas se distingue de la exponencial","catalogo.H2H_ATRIBUTOS (regla 1)"),
 ("forma","form_* jornada 1","imputar","cold-start real de inicio de temporada","mediana dev + flag_coldstart"),
 ("forma","form_*_ewm (decaimiento exponencial)","mantener","la ventana rectangular no es la unica hipotesis de olvido: pondera igual el 1o y el 5o partido y descarta el 6o de golpe","semivida DECLARADA, no ajustada: FORMA_SEMIVIDA=3"),
 ("forma","nivel (home_/away_) solo del nucleo","recortar","en puntos, goles y peligro el nivel absoluto discrimina; en el resto la resta agota la informacion","catalogo.FORMA_NUCLEO (regla 2)"),
 ("forma","form_shot_* (calidad de ocasion)","mantener","la distribucion de remates no se deduce del volumen ni del xG total: dice como se reparte el peligro entre los disparos","solo por ventana retrospectiva; en crudo el bloque es fuga (columns.LEAKAGE_RAW)"),
 ("forma","form_shot_* sin volcado de tiros","tirar","bloque opcional ausente: columnas enteras a NaN, sin variacion que imputar ni que transformar","poda por constancia en fit(dev), ANTES de la imputacion"),
 ("forma","form_opp_elo_z_*","mantener","corrige el sesgo de calendario: el mismo balance no vale igual segun a quien se enfrento","fuerza tipificada media del rival reciente, sin parametros ajustados"),
 ("forma","fin_resid / def_resid","no construir","residuo vs xG = suerte/varianza, no habilidad: por definicion la parte NO reproducible del rendimiento","catalogo (regla 5)"),
 ("fuerza","strength_attack / strength_defence","mantener","el Elo funde en un escalar las dos maneras de ser bueno -marcar mucho y encajar poco- y el empate depende de esa distincion","modelo de goles Maher/Dixon-Coles; semivida 180 d DECLARADA, no ajustada"),
 ("fuerza","strength_p_home/draw/away","mantener","pronostico 1X2 completo sin mercado ni estimador ajustado: entra como variable y como peldano de referencia de la fase 5","probabilidades renormalizadas; suma 1 verificada en qa_features"),
 ("fuerza","strength_exp_goals","no construir","los goles esperados son exactamente lo que la terna de probabilidades resume","catalogo.FUERZA_PRONOSTICO (regla 4)"),
 ("fuerza","strength_* sin recorrido en la categoria","imputar","el bloque describe un ENFRENTAMIENTO: si a uno de los dos clubes le falta historia, el partido entero se queda sin el","mediana dev + flag_no_strength"),
 ("elo","home_elo_z / away_elo_z","mantener como par","el par abarca a la vez la fuerza relativa de cada equipo y su diferencia; emitir tambien la resta es entregar una columna deducible","catalogo.ELO (regla 2)"),
 ("elo","home_elo / away_elo (nivel crudo)","apartar","los dos tramos del empalme de ClubElo no son comparables en nivel: en la matriz seria un escalon de fuerza simultaneo para los veinte clubes","se queda en el master como columna descriptiva; el modelo consume la z"),
 ("elo","elo_exp_home","no construir","duplicado determinista de la diferencia de Elo (r=1)","catalogo (regla 3)"),
 ("clasificacion","europe_unreachable","no construir","complemento casi determinista de `points_to_europa`","catalogo (regla 3)"),
 ("clasificacion","stakes_none por lado","no construir","la pregunta real es si el partido es intrascendente para LOS DOS: el producto","catalogo.TRASCENDENCIA_PARTIDO"),
 ("psico","n_suspension / post_big_fav_loss","no construir","señal nula: MI≈0.001-0.004, orden del ruido de muestreo","catalogo (regla 5)"),
 ("contexto","kickoff_hour","no construir","la franja es la misma informacion agrupada en las tres categorias que significan algo","catalogo.CONTEXTO_CATEGORICO (regla 1)"),
 ("h2h","2 constantes todo-cero","tirar","varianza nula: no informan","drop por constancia en fit(dev) (D.3.a)"),
 ("varias","colineales |r|>0.95","tirar","redundancia que solo se ve en los datos; se queda el de mayor MI×cobertura","greedy, protege mkt_p_* (D.3.b)"),
 ("mercado","mkt_p_*","mantener","variable y linea base a la vez: el consenso del mercado sin margen","terna protegida de la poda"),
 ("mercado","mkt_p_b365_* / odds_ps_*","apartar","referencia de evaluacion, no variable: el consenso ya promedia esas casas","columns.SERVICIO, fuera de feature_cols"),
 ("mercado","mkt_overround_* / mkt_spread_* / sharp_div_*","mantener","margen y desacuerdo entre casas: señal que la normalización de mkt_p_* destruye","estructura del precio, régimen semanal"),
 ("categóricas","position_zone / playstyle / kickoff_slot","encoding","orden natural -> ordinal (no one-hot: preserva monotonía)","ZONE_ORDER / PLAYSTYLE_ORDER / SLOT_ORDER"),
 ("varias","continuas con |skew|>1","transformar","colas largas -> potencia Yeo-Johnson (λ MLE en dev, congelado)","transform=True (D.4)"),
], columns=["familia","columna","accion","motivo","parametro"])
pd.set_option("display.max_colwidth", 78)
print(decisiones.to_string(index=False))

### D.2 · Imputación por causa (no un `fillna(0)` global)
Cada `NaN` se imputa **según su causa** —no con un `df.fillna(0)` global que
trataría todos los huecos por igual—, con la constante aprendida en `fit(dev)` y **congelada**, acompañada de un
**flag binario por causa** para que el modelo distinga "valor imputado" de "valor
real". Cuatro estrategias, y el criterio que las separa es **cuánta información
contiene el propio hueco**: hay huecos que dicen algo del sujeto (un ascendido es
débil), huecos cuyo neutro es evidente (nunca hubo enfrentamiento), huecos que tienen
un valor exacto y deducible (la liga aún no ha empezado) y huecos que obligan
sencillamente a estimar.

- **`weak_prior`** — ascendidos sin valor t−1. Se imputa el **cuantil 0.10 de dev**
  (cola inferior), no la mediana: un ascendido es estructuralmente débil. La
  imputación actúa **directamente sobre las columnas que viajan** —el par
  `home_squad_value` / `away_squad_value` y `age_diff`—, sin ningún recálculo
  posterior de derivadas. Que no haga falta ese recálculo es consecuencia del
  catálogo: al no declararse ni la diferencia ni el log-ratio del valor de plantilla,
  no queda ninguna columna aguas abajo que pudiera quedarse desincronizada respecto de
  las bases imputadas. La regla general que esto ilustra: cuando una derivada
  determinista viaja junto a sus bases, la imputación deja de ser una operación local y
  pasa a exigir un orden de recálculo; suprimir la derivada suprime el problema entero.
- **`zero`** — ausencia **real** cuyo neutro es 0: sin historial de enfrentamientos
  (H2H) o sin partidos disputados en ese campo (venue). No falta el dato: es que el
  suceso no ha ocurrido, y su recuento correcto es cero.
- **`known`** — el hueco tiene un valor **exacto y deducible**, no estimable. El bloque
  de trascendencia se calcula sobre la clasificación *previa* al partido, así que en la
  jornada 1 llega vacío; pero ese vacío describe una situación perfectamente
  determinada —el marcador de la liga a cero—: todas las distancias a un corte valen 0
  porque los veinte equipos empatan a cero puntos, los tres indicadores de objetivo
  decidido valen 0 porque nada puede estar decidido tras cero jornadas y `matches_left`
  vale 38 porque queda el calendario íntegro. Imputar aquí la mediana de dev sería
  sustituir un dato cierto por una estimación, y encima sesgada: la mediana se calcula
  sobre partidos de mitad de temporada, cuando ya hay distancias abiertas.
- **`neutral`** — mediana de dev ("equipo promedio, aún sin señal"). Es la estrategia
  **de reserva**, la que actúa cuando el hueco sí obliga a estimar: cold-start de forma
  y de posición en la tabla, la fuerza latente de un partido en el que alguno de los dos
  clubes no acumula todavía recorrido en la categoría, huecos de época (cuotas de cierre
  16–19, bajas 16/17) y cualquier falta de cobertura del contexto.

Ningún estadístico de imputación cruza el corte train/holdout: todos salen de dev.
Los flags se calculan del `df` de entrada **antes** de podar, de modo que
sobreviven aunque su columna fuente se elimine luego por colinealidad. El bloque de
trascendencia no estrena flag propio, y la omisión es deliberada: su hueco y el de la
forma reciente son el **mismo suceso** —la temporada aún no ha empezado—, así que
`flag_coldstart` ya lo marca y un flag adicional sería una copia exacta de aquel, que
solo añadiría colinealidad a la matriz.

La **fuerza latente**, en cambio, sí estrena flag —`flag_no_strength`—, y el contraste
con el caso anterior es exactamente el criterio que decide cuándo un flag está
justificado: **que el suceso no se deduzca de ningún otro**. El hueco de la fuerza
aparece cuando alguno de los dos clubes no acumula tres partidos efectivos en la
categoría, y ese suceso no lo marca ninguno de los cinco flags previos:
`flag_coldstart` se activa para **todos** los equipos en la jornada 1 de cada temporada
—incluidos los que llevan una década en Primera—, y `flag_no_squadvalue` se mantiene
activo para un ascendido durante el **año entero**, mucho después de que el bloque de
fuerza haya empezado a publicar. Sin flag propio, el modelo recibiría el bloque completo
sustituido por la mediana de la liga —la descripción de un partido medio allí donde no
hay descripción— sin nada que se lo advirtiera, y el efecto sería peor que el ruido:
un ascendido en sus primeras jornadas entraría al modelo con la fuerza de un equipo
promedio, que es precisamente lo contrario de lo que se sabe de él.

Los cinco flags se conservan como **entradas del modelo**, no solo como diagnóstico. La
razón es que el hecho de que un dato falte es, en este dominio, informativo por sí mismo:
`flag_no_squadvalue` identifica a los recién ascendidos, `flag_coldstart` marca el arranque
de temporada y `flag_no_odds_close` separa la época sin cuotas de cierre. Son tres
regularidades reales del calendario y de la cobertura de las fuentes, y esconderlas al
modelo no las haría desaparecer: solo le impediría condicionar sobre ellas.

In [ ]:
# --- D.2 · Imputación por causa (materializada en preprocess.fit) ----------
# preprocess.py aprende TODO en dev y lo congela. Cuatro estrategias por causa,
# ordenadas de mas a menos informativo el hueco: prior debil, cero real, valor cierto
# y, solo cuando no queda mas remedio, la mediana estimada.
import importlib; importlib.reload(pp)

# El régimen publicable es el que se audita aquí: todo lo que sigue (imputación,
# poda, transformación, escalado) describe el modelo que se ejecuta cada semana.
REGIMEN = "semanal"
st = pp.fit(dev, regime=REGIMEN)
print(f"régimen ajustado: {st.regime}  "
      f"({st.meta['n_regime_excluded']} columnas fuera por no ser conocibles al predecir)")
sc = st.meta["impute_strategy_counts"]
print("estrategia de imputación por columna (aprendida en dev):")
print(f"  weak_prior (ascendidos, cuantil {int(pp._WEAK_PRIOR_Q*100)}%): {sc['weak_prior']} cols")
for c, v in st.meta["weak_prior_values"].items():
    print(f"      {c:20s} <- {v:,.1f}")
print(f"  zero    (ausencia real h2h/venue):          {sc['zero']} cols")
print(f"  known   (arranque de temporada, valor cierto): {sc['known']} cols")
# el valor cierto no se elige: se deduce. Se listan los distintos que aparecen para
# comprobar de un vistazo que solo hay dos -el cero de distancias y banderas y los
# 38 partidos del calendario- y que ninguno procede de un estadistico de dev.
print(f"      valores distintos imputados: {sorted(set(st.meta['known_values'].values()))}")
print(f"  neutral (mediana dev; lo que sí hay que estimar): {sc['neutral']} cols")
print(f"\nflags de faltante por CAUSA ({len(st.flag_cols)}): {list(st.flag_cols)}")

### D.3 · Encoding de categóricas + poda por reglas
- `position_zone` → **ordinal** {descenso < mid_table < europa < champions}.
- `playstyle` → **ordinal** {defensivo < equilibrado < ofensivo}. Se codifica
  ordinal (no one-hot): el eje tiene orden natural, así se preserva la monotonía
  con una sola columna y sin inflar la dimensionalidad. Los niveles válidos se
  fijan en `fit(dev)`; el holdout/upcoming no puede introducir categorías nuevas.
- `kickoff_slot` → **ordinal** {mediodia < tarde < noche}. El orden es el del reloj, de
  manera que la codificación conserva la única relación que la variable tiene y no
  inventa ninguna. La hora exacta de saque ya no se construye: la franja es esa misma
  información agrupada en las tres categorías que significan algo, y mantener las dos
  era ofrecer dos parametrizaciones del mismo atributo (regla 1 del catálogo).

La poda de redundancia se ejecuta dentro de `fit`, y a estas alturas del proyecto le
quedan **dos reglas**: (a) columnas **constantes** y (b) columnas **colineales**
`|r|>0.95`, por *greedy* que conserva la de mayor `MI×cobertura` y que protege la terna
`mkt_p_*`, que es a la vez variable y línea base (las ternas de Bet365 y de Pinnacle no
entran en la poda porque no son candidatas a feature: viajan como servicio de
evaluación).

Aquí había antes tres reglas más —el duplicado determinista `elo_exp_home`, los
residuos-suerte `fin_resid`/`def_resid` y las de señal nula
`n_suspension`/`post_big_fav_loss`—. Ya no están, y su desaparición **no es un cambio de
criterio sino un cambio de lugar**: esas columnas no se podan al final porque no llegan a
construirse. `catalogo.py` es ahora la única fuente declarativa de qué atributo sale a la
matriz y con qué parametrización, y las tres decisiones viven allí como exclusiones
razonadas (reglas 1, 2 y 5). La diferencia importa: una columna que se calcula y se tira
es trabajo muerto que atraviesa medio pipeline y ensucia todo lo que toca por el camino
—los inventarios, los mapas de faltantes, la trazabilidad—, mientras que una columna no
declarada sencillamente no existe en ningún punto.

Lo que se queda en `fit` es exactamente lo que **no se puede decidir de antemano**: si una
columna resulta constante y si dos columnas resultan redundantes son preguntas que solo
contestan las filas de dev.

In [ ]:
# --- D.3 · Encoding de categóricas + poda por reglas -----------------------
print("mapas ordinales (fijados en fit; el holdout no puede crear niveles):")
print("  position_zone :", pp.ZONE_ORDER)
print("  playstyle     :", pp.PLAYSTYLE_ORDER, "  (ORDINAL, no one-hot)")
print("  kickoff_slot  :", pp.SLOT_ORDER, "        (orden del reloj)")
m = st.meta
print(f"\npoda por reglas -> {len(st.drop_cols)} columnas eliminadas:")
print(f"  constantes ({m['n_const']}): {m['const_cols']}")
print(f"  colineales ({m['n_collinear']}, |r|>{m['corr_threshold']}) - se queda el de mayor MI x cobertura:")
for a, b, r in m["collinear_pairs_kept_dropped"][:8]:
    print(f"      r={r:.3f}  KEEP {a:26s} DROP {b}")
print(f"      ... (+{max(0, m['n_collinear']-8)} pares mas)")

# Las reglas que antes tiraban columnas por su NOMBRE ya no viven aqui. Se
# comprueba que efectivamente no queda rastro de ellas en la matriz: si alguna de
# estas columnas reapareciera, significaria que un modulo de features ha vuelto a
# emitir algo que el catalogo no declara, y la poda de este punto ya no estaria
# para taparlo.
_EXCLUIDAS_EN_CATALOGO = ["elo_exp_home", "fin_resid", "def_resid",
                          "n_suspension", "post_big_fav_loss", "kickoff_hour",
                          "squad_value_diff", "squad_value_logratio"]
_rastro = sorted({c for c in feat.columns
                  for t in _EXCLUIDAS_EN_CATALOGO if t in c})
print("\nexcluidas en catalogo.py (no se construyen, no se podan aqui):")
print(f"  {', '.join(_EXCLUIDAS_EN_CATALOGO)}")
print(f"  rastro en la matriz: {_rastro or 'ninguno'}")
assert not _rastro, f"columnas excluidas por catalogo que han reaparecido: {_rastro}"


### D.4 · Simetrización de colas (Yeo-Johnson)
Varias variables continuas arrastran colas muy largas (asimetría fuerte): cuotas,
días de descanso, valor de las bajas, valor de plantilla... Para que el escalado
posterior no quede dominado por unos pocos valores extremos —y para acercar las
marginales a algo más simétrico— se aplica una **transformación de potencia
Yeo-Johnson**: por cada columna elegible se ajusta su parámetro `λ` por máxima
verosimilitud **solo en dev** y se **congela**; `apply()` reutiliza ese mismo `λ`
en holdout y en `features_upcoming`, garantizando paridad exacta train↔inferencia.

**Qué se transforma.** Solo continuas con **asimetría fuerte** (`|skew|>1`); se
**excluyen** las probabilidades acotadas en `[0,1]` —las del mercado (`mkt_p_*`, ya
interpretables) y las del modelo de goles (`strength_p_*`)— y las categóricas
codificadas. La exclusión no se declara por una lista de nombres sino por el rango
empírico de la columna en dev, de modo que cualquier probabilidad futura queda fuera
sin editar nada. Yeo-Johnson admite valores negativos, así que también
vale para los diferenciales (`*_diff`).

**Guarda anti-colapso.** Un `λ` muy negativo sobre una variable de gran magnitud
(p. ej. Elo ≈ 1500) hace que `(x+1)^λ` caiga a *denormales* y colapse la columna a
casi-constante: el `skew` "mejora" a 0 pero se aniquila la varianza. Para evitarlo,
la transformación se **rechaza** si no preserva el número de valores distintos
(resolución ≥ 0.99). Por eso el **Elo NO se transforma** aunque el MLE proponga un
λ: la guarda lo detecta y lo deja intacto. Es una capa activable (`transform=True`
por defecto).

In [ ]:
# --- D.4 · Simetrización de colas (Yeo-Johnson) ----------------------------
# λ ajustado por MLE SOLO en dev y congelado; se aplica antes de escalar. Solo
# continuas con |skew|>1; excluye probabilidades [0,1] y categóricas. La guarda
# anti-colapso rechaza el λ que no preserve los valores distintos (p. ej. Elo).
rows = st.meta["transform_skew_before_after"]        # (col, λ, skew_antes, skew_después)
print(f"columnas simetrizadas: {st.meta['n_transformed']}  "
      f"(Elo excluido por la guarda anti-colapso)")
imp = sorted([(c, b, a) for c, _, b, a in rows], key=lambda t: -abs(t[1]))
print(f"\n{'columna':30s} {'skew_in':>9s} {'skew_out':>9s}")
for c, b, a in imp[:12]:
    print(f"  {c:28s} {b:+9.2f} {a:+9.2f}")
mb = np.mean([abs(b) for _, b, _ in imp]); ma = np.mean([abs(a) for _, _, a in imp])
print(f"\n|skew| medio: {mb:.2f} -> {ma:.2f}   |skew| máx tras transformar: "
      f"{max(abs(a) for _, _, a in imp):.2f}")

### D.5 · Escalado
Se **materializa** aquí: estandarización (μ/σ) aprendida **solo en dev** y sobre los
valores **ya imputados/recalculados**, de modo que dev queda exactamente media 0 /
desviación 1. Solo se escalan las columnas **continuas** (`nunique>2`); binarias y
flags se dejan intactas. `apply()` reutiliza esas μ/σ congeladas en holdout y en
`features_upcoming`, sin recomputarlas. Es desactivable sin tocar el resto del
pipeline (útil si 05 usa árboles/boosting, que no lo necesitan).

In [ ]:
# --- D.5 · Escalado (estandarización sobre valores imputados) --------------
# μ/σ aprendidas SOLO en dev y sobre los valores YA imputados/recalculados, de
# modo que dev queda exactamente media 0 / desv 1. Binarias y flags NO se escalan.
print(f"columnas escaladas: {st.meta['n_scaled']} de {len(st.feature_cols)} features")
print("ejemplo (μ, σ de dev):")
for c in list(st.scale_mean)[:5]:
    print(f"  {c:24s} μ={st.scale_mean[c]:9.3f}  σ={st.scale_std[c]:9.3f}")

### D.6 · El módulo `preprocess.py` y su aplicación
El preprocesado es una **función reproducible**, no una secuencia de celdas. Vive
en `scripts/features/preprocess.py`:
```
fit(dev)          -> PreprocessState   (serializable: drop_cols, valores + estrategia
                     de imputación, flags, mapas ordinales, μ/σ de dev)
apply(df, state)  -> df_prep           (idéntico para dev, holdout y upcoming)
```
`PreprocessState.save/load` (JSON) persiste el estado para que `build_scoring` →
`features_upcoming` pase por **exactamente el mismo** `apply`, garantizando paridad
train↔scoring. La Parte E lo verifica (mismo esquema, refit determinista).

In [ ]:
# --- D.6 · Aplicación del módulo (fit dev -> transform dev/holdout) --------
pp.report(st)
Xdev  = pp.apply(dev, st)
Xhold = pp.apply(holdout, st)
fc = [c for c in st.feature_cols if c in Xdev.columns]
print(f"\ndev_prep     = {Xdev.shape}")
print(f"holdout_prep = {Xhold.shape}")
print(f"NaN en features: dev={int(Xdev[fc].isna().sum().sum())}  "
      f"holdout={int(Xhold[fc].isna().sum().sum())}")
print("esquema idéntico dev/holdout:", list(Xdev.columns) == list(Xhold.columns))

---
## Parte E — QA del preprocesado

Igual que `qa_features` blinda las features, esta parte blinda la salida del
preprocesado con invariantes duros. Es el "sello" de que la matriz de modelado
está limpia y es reproducible.

### E.1–E.6 · Invariantes de salida
1. **Sin NaN residual** (o cada NaN restante justificado y con su flag).
2. **Paridad de esquema** dev/holdout/upcoming: mismas columnas, mismo orden,
   mismos dtypes (esto es lo que garantiza que la inferencia no rompa).
3. **No-leak**: `fit` solo vio `dev`; ninguna constante/encoder/escala se calculó
   con holdout. (Chequeo: `fit(dev)` re-ejecutado da los mismos `drop_cols` y
   `feature_cols` — determinismo.)
4. **Target intacto**: `y`/`y_num` sin tocar; conteos por split correctos
   (dev 3420, holdout 380).
5. **Baseline de mercado calculable** desde el artefacto en escala cruda: las
   probabilidades implícitas sobreviven al preprocesado como número a batir.
6. **Coherencia del régimen**: el estado declara el régimen con el que se ajustó y,
   bajo el semanal, ninguna feature ni flag procede del bloque de cierre. El régimen
   deja así de ser una intención del código para ser una **propiedad verificada del
   artefacto**.

**Persistencia con un único escritor.** Los cuatro artefactos —
`features_prep_<régimen>.parquet` (dev y holdout en un solo archivo, distinguidos por
la columna `split`) y `preprocess_state_<régimen>.json`, uno de cada por régimen— los
escribe **`scripts/features/build_preprocess.py`**, no este notebook. La razón es de
trazabilidad: dos rutas de escritura del mismo artefacto no pueden garantizar que
produzcan lo mismo, así que el notebook *muestra y audita* y el orquestador *ejecuta y
persiste*. La celda siguiente importa ese orquestador y comprueba las seis invariantes
sobre exactamente la matriz que el script escribirá. El holdout se transforma pero
**sigue sellado** (no ha decidido nada).

In [ ]:
# --- E.1-E.6 · Invariantes de salida + persistencia ------------------------
fc = [c for c in st.feature_cols if c in Xdev.columns]
assert Xdev[fc].isna().sum().sum() == 0,  "E.1 dev con NaN"          # E.1 sin NaN
assert Xhold[fc].isna().sum().sum() == 0, "E.1 holdout con NaN"
assert list(Xdev.columns) == list(Xhold.columns), "E.2 esquema distinto"  # E.2 paridad
st2 = pp.fit(dev, regime=REGIMEN)                                    # E.3 determinismo
assert st2.drop_cols == st.drop_cols and st2.feature_cols == st.feature_cols, "E.3"
assert len(Xdev) == len(dev) and len(Xhold) == len(holdout), "E.4 filas"   # E.4 conteos
print("E.1 sin NaN OK   E.2 esquema idéntico OK   E.3 refit determinista OK   "
      f"E.4 filas {len(Xdev)}/{len(Xhold)} OK")

# E.5-E.6 · PERSISTENCIA: la hace el orquestador, no el notebook.
# `build_preprocess()` repite fit(dev) -> apply(todo) y verifica dentro las mismas
# invariantes E.1-E.4, mas la E.5 (los TRES baselines -consenso, Pinnacle y Bet365-
# siguen siendo calculables desde el artefacto, en escala cruda, y ninguno de ellos
# se ha colado como feature) y la E.6 (coherencia del regimen). Se importa
# aqui para auditar EXACTAMENTE la matriz que el script escribira: si el notebook
# escribiese su propia copia habria dos rutas de escritura del mismo fichero sin
# garantia de que coincidan.
import build_preprocess as bp
importlib.reload(bp)

Xall, st_orq = bp.build_preprocess(feat, regime=REGIMEN)
assert st_orq.regime == REGIMEN, "E.6 el estado del orquestador es de otro regimen"
assert st_orq.feature_cols == st.feature_cols, "el estado del orquestador difiere del del notebook"
assert st_orq.drop_cols == st.drop_cols, "las podas del orquestador difieren de las del notebook"
assert list(Xall.columns) == list(Xdev.columns), "E.2 esquema del artefacto distinto del auditado"
assert len(Xall) == len(feat), "E.4 el artefacto no cubre todas las filas"
assert not C.weekly_unavailable(list(st_orq.feature_cols)
                                + list(st_orq.flag_cols.values())), "E.6 columna de cierre"
print("E.5 baselines consenso, Pinnacle y Bet365 OK   E.6 regimen coherente OK   |  "
      "el estado del orquestador coincide con el auditado")
print(f"matriz de modelado ({REGIMEN}): {Xall.shape}  ({len(st.feature_cols)} features, "
      f"{st.meta['n_scaled']} escaladas, {len(st.flag_cols)} flags)")

# El regimen retrospectivo se ajusta con el MISMO protocolo y solo anade el bloque de
# cierre; su diferencia con el semanal es la informacion que el mercado incorpora
# entre la apertura y el cierre, no un cambio de metodo.
Xret, st_ret = bp.build_preprocess(feat, regime="retrospectivo")
print(f"matriz de modelado (retrospectivo): {Xret.shape}  "
      f"({len(st_ret.feature_cols)} features, {len(st_ret.flag_cols)} flags)")
print(f"diferencia de features: {len(st_ret.feature_cols) - len(st.feature_cols):+d}")

print("\nPersistencia (ruta unica, fuera del notebook):")
print("    python scripts/features/build_preprocess.py         # escribe los dos regimenes")
print("    python scripts/features/build_preprocess.py --dry   # audita sin escribir")
for reg in pp.REGIMES:
    print(f"    -> {bp.prep_path(reg)}")
    print(f"    -> {bp.state_path(reg)}")

---
### Resumen de decisiones (listo para la memoria)
Partiendo del conjunto completo de columnas —candidatas a feature más identidad y
objetivo, cuyo recuento imprime A.1—,
`preprocess.fit(dev, regime)` produce un conjunto de features y un estado congelado y
serializable por cada régimen de información. Las cifras que siguen son las del
**régimen semanal**, el que se publica; el retrospectivo aplica el mismo protocolo
sobre el mismo material más el bloque de cierre.

- **Recorte en origen, no al final**: el conjunto candidato lo fija `catalogo.py`, que
  declara para cada atributo qué lados se emiten —par, diferencia, una de las dos o una
  sola columna de partido—. Las columnas que antes se construían y se podaban después
  (la diferencia de valor de plantilla y su log-ratio, la expectativa Elo, los residuos
  de suerte, el desglose de bajas por tipo, los goles esperados agregados del modelo de
  fuerza) hoy **no se construyen**. El criterio no ha cambiado: ha cambiado el sitio
  donde se aplica, y con él el coste, porque una columna que se calcula para tirarla
  atraviesa la extracción, la consolidación, la imputación y el escalado antes de morir.
- **Régimen de información**: el ajuste se restringe a lo conocible al emitir la
  predicción. Las columnas derivadas del cierre —y el flag que las acompaña— quedan
  fuera del semanal *antes* de la poda, porque la poda colineal las enfrentaría con su
  gemela de apertura y podría dejar al modelo publicado sin ninguna de las dos.
- **Imputación por causa** (nada de `fillna(0)` ciego): `weak_prior` para los
  ascendidos (cuantil 10 % de dev), `zero` para la ausencia real de historial
  (h2h/venue), `known` para el hueco de valor deducible (jornada 1 del bloque de
  trascendencia) y `neutral` (mediana dev) para el cold-start de forma y los huecos de
  época, cada una con su **flag por causa**, conservado como entrada del modelo.
- **Poda por reglas, reducida a dos**: constantes (las columnas h2h todo-cero de un
  régimen sin historial) y colineales `|r|>0.95`, resueltos por *greedy* que conserva el
  de mayor `MI×cobertura`, con la terna de probabilidad implícita del consenso
  (`mkt_p_*`) protegida. No queda ninguna regla de "señal nula" ni de duplicado
  determinista: esos casos se resuelven ahora en el catálogo, que es donde se decide qué
  existe.
- **Las lecturas de la memoria en competencia**: cada atributo de forma entra en ventana
  rectangular y en decaimiento exponencial de semivida declarada, y junto a ellas viajan
  el acumulado de temporada, la forma restringida al escenario y la dureza del
  calendario recorrido. Ninguna se privilegia a priori: es la selección hacia delante
  del 05, sobre bloque separado, la que arbitra el solape.
- **Encoding ordinal** de `position_zone` y `playstyle` (orden natural).
- **Simetrización Yeo-Johnson** de las continuas de cola larga (`λ` por MLE en dev,
  congelado), con guarda anti-colapso que excluye el Elo.
- **Escalado** de las continuas (μ/σ de dev); binarias y flags intactas.
- **Mercado mantenido** como feature: una sola terna de probabilidad implícita —el
  consenso— más la **estructura del precio** (margen, desacuerdo entre casas y
  divergencia con Pinnacle), que recupera lo que la normalización de `mkt_p_*` destruye.
  El precio se expone además aparte (`baselines_report`) como número a batir: el
  **mejor** de los tres baselines, consenso, Pinnacle y Bet365, comparados sobre la
  intersección de filas evaluables. Pinnacle y Bet365 son solo referencia y se
  reconstruyen desde las cuotas crudas de servicio, que viajan en el artefacto fuera de
  `feature_cols`.

Salida: `features_prep_<régimen>.parquet` + `preprocess_state_<régimen>.json` para
cada uno de los dos regímenes, escritos por
`python scripts/features/build_preprocess.py`. El holdout 25/26 solo se transformó,
jamás se ajustó (contrato anti-fuga intacto).

---
### El estado congelado y el estado del pliegue

El estado que esta parte deja escrito en disco es el **estado de inferencia**: el que
`build_predict.py` carga para puntuar la jornada por jugar. Tiene que ser fijo y estar
aprendido sobre todo el desarrollo, porque la alternativa —reajustarlo cada semana en
producción— haría que dos predicciones emitidas en semanas distintas no fueran
comparables entre sí, y una probabilidad que cambia porque ha cambiado la escala no
informa de nada.

Medir, sin embargo, es otro problema. `fit` aprende medianas de imputación, los
parámetros λ de la simetrización y —sobre todo— la selección de la poda colineal, que
es supervisada porque arbitra por `MI × cobertura`. Usar ese único estado dentro de la
validación no filtraría el resultado del partido que se predice, pero sí filtraría la
**distribución futura**: un pliegue de 2019 estaría transformado con medianas de nueve
temporadas y, peor aún, con una selección de variables que ya sabe cuáles acabarían
siendo informativas en 2024.

Por eso el mismo `fit` de esta parte se vuelve a ejecutar dentro de cada pliegue del
protocolo, sobre la ventana que precede al partido evaluado y solo sobre filas de
desarrollo. No son dos preprocesados distintos —es el mismo código, las mismas
decisiones y el mismo régimen—, sino el mismo procedimiento aplicado a dos ventanas
distintas porque responde a dos preguntas distintas: *con qué transformo lo que aún no
ha ocurrido* y *con qué habría transformado lo que ya ocurrió, sin mirar más allá*.

Como efecto secundario, ese reajuste corrige la deriva que A.5 y C.3 dejan a la vista:
la mediana neutral deja de ser la de nueve temporadas y pasa a ser la de la ventana
inmediatamente anterior, de modo que un hueco de 2024 se rellena con un valor de 2024.
`protocol.imputation_drift` cuantifica esa deriva sobre los estados realmente
ajustados, que es la comprobación empírica que esta decisión pedía.
